In [1]:
# V11 — VIIRS Event Formation & Dataset Quality
# SIH 2026
# Goal: Build a physically and temporally coherent thermal-event dataset

import os
import warnings
import time

import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

EARTH_RADIUS_KM = 6371.0088

# Raw VIIRS dataset
VIIRS_FILE = "viirs-snpp_2024_India.csv"

# Current POC period
START_DATE = "2024-01-01"
END_DATE   = "2024-01-31"

# Candidate spatial association scales
SPATIAL_RADII_KM = [
    0.375,
    0.500,
    0.750,
    1.000
]

# Maximum temporal gap for continuation
TEMPORAL_GAP_DAYS = 3

# Output directory
OUTPUT_DIR = "v11_event_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("V11 — VIIRS Event Formation & Dataset Quality")
print("=" * 55)
print(f"Input file        : {VIIRS_FILE}")
print(f"Date range        : {START_DATE} → {END_DATE}")
print(f"Candidate radii   : {SPATIAL_RADII_KM} km")
print(f"Temporal gap      : {TEMPORAL_GAP_DAYS} days")
print(f"Output directory  : {OUTPUT_DIR}")

V11 — VIIRS Event Formation & Dataset Quality
Input file        : viirs-snpp_2024_India.csv
Date range        : 2024-01-01 → 2024-01-31
Candidate radii   : [0.375, 0.5, 0.75, 1.0] km
Temporal gap      : 3 days
Output directory  : v11_event_dataset


In [2]:
# ---------------------------------------------------------
# LOAD RAW VIIRS DATA
# ---------------------------------------------------------

start = time.time()

viirs = pd.read_csv(
    VIIRS_FILE,
    parse_dates=["acq_date"]
)

print(f"Loaded {len(viirs):,} raw detections")
print(f"Columns: {list(viirs.columns)}")
print(f"Load time: {time.time() - start:.2f} seconds")

viirs.head()

Loaded 552,312 raw detections
Columns: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type']
Load time: 1.56 seconds


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,17.1696,79.9904,332.0500,0.6300,0.7200,2024-01-01,702,N,VIIRS,n,2,294.5400,3.6800,D,0
1,18.2435,83.9555,333.7100,0.3300,0.5600,2024-01-01,702,N,VIIRS,n,2,296.6600,1.7500,D,0
2,18.7408,83.9803,340.1800,0.3200,0.5500,2024-01-01,702,N,VIIRS,n,2,299.8600,4.2400,D,0
3,18.7094,83.4534,327.8500,0.3500,0.5700,2024-01-01,702,N,VIIRS,n,2,299.3400,2.8700,D,0
4,18.7087,83.4500,329.7000,0.3500,0.5700,2024-01-01,702,N,VIIRS,n,2,296.8100,2.0400,D,0


In [3]:
# ---------------------------------------------------------
# RAW DATA VALIDATION
# ---------------------------------------------------------

print("Dataset shape:", viirs.shape)
print()

print("Missing values:")
print(viirs.isna().sum())
print()

print("Date range:")
print(viirs["acq_date"].min(), "→", viirs["acq_date"].max())
print()

print("Day/Night:")
print(viirs["daynight"].value_counts(dropna=False))
print()

print("Satellite:")
print(viirs["satellite"].value_counts(dropna=False))
print()

print("Type:")
print(viirs["type"].value_counts(dropna=False))
print()

print("Confidence:")
print(viirs["confidence"].value_counts(dropna=False))

Dataset shape: (552312, 15)

Missing values:
latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
dtype: int64

Date range:
2024-01-01 00:00:00 → 2024-12-31 00:00:00

Day/Night:
daynight
D    395587
N    156725
Name: count, dtype: int64

Satellite:
satellite
N    552312
Name: count, dtype: int64

Type:
type
0    483614
2     67587
3      1097
1        14
Name: count, dtype: int64

Confidence:
confidence
n    435018
l    103437
h     13857
Name: count, dtype: int64


In [4]:
# ---------------------------------------------------------
# FILTER POC INPUT
# ---------------------------------------------------------

night = viirs[
    (viirs["acq_date"] >= START_DATE) &
    (viirs["acq_date"] <= END_DATE) &
    (viirs["daynight"].str.lower() == "n")
].copy()

night = night.sort_values(
    ["acq_date", "latitude", "longitude"]
).reset_index(drop=True)

print(f"Nighttime detections: {len(night):,}")
print()

print("Date coverage:")
print(night["acq_date"].min(), "→", night["acq_date"].max())

print()
print("Coordinate range:")
print(f"Latitude : {night['latitude'].min():.5f} → {night['latitude'].max():.5f}")
print(f"Longitude: {night['longitude'].min():.5f} → {night['longitude'].max():.5f}")

Nighttime detections: 13,609

Date coverage:
2024-01-01 00:00:00 → 2024-01-31 00:00:00

Coordinate range:
Latitude : 8.24339 → 34.63008
Longitude: 68.57577 → 97.07681


In [5]:
# ---------------------------------------------------------
# VERIFY JANUARY NIGHTTIME INPUT
# ---------------------------------------------------------

jan_night = viirs[
    (viirs["acq_date"].dt.year == 2024) &
    (viirs["acq_date"].dt.month == 1) &
    (viirs["daynight"] == "N")
].copy()

jan_night = jan_night.reset_index(drop=True)

print("January 2024 nighttime VIIRS input")
print("=" * 45)

print(f"Detections : {len(jan_night):,}")
print(f"Dates      : {jan_night['acq_date'].min().date()} → {jan_night['acq_date'].max().date()}")

print("\nConfidence:")
print(jan_night["confidence"].value_counts())

print("\nType:")
print(jan_night["type"].value_counts())

print("\nCoordinates:")
print(
    f"Latitude  : {jan_night['latitude'].min():.5f} → "
    f"{jan_night['latitude'].max():.5f}"
)
print(
    f"Longitude : {jan_night['longitude'].min():.5f} → "
    f"{jan_night['longitude'].max():.5f}"
)

January 2024 nighttime VIIRS input
Detections : 13,609
Dates      : 2024-01-01 → 2024-01-31

Confidence:
confidence
n    13599
h       10
Name: count, dtype: int64

Type:
type
2    8169
0    5338
3     102
Name: count, dtype: int64

Coordinates:
Latitude  : 8.24339 → 34.63008
Longitude : 68.57577 → 97.07681


In [6]:
# ---------------------------------------------------------
# DUPLICATE / DATA-INTEGRITY CHECK
# ---------------------------------------------------------

print("Duplicate checks")
print("=" * 45)

# Exact duplicate rows
exact_duplicates = jan_night.duplicated().sum()

# Duplicate spatial-temporal observations
spatiotemporal_duplicates = jan_night.duplicated(
    subset=[
        "latitude",
        "longitude",
        "acq_date",
        "acq_time"
    ]
).sum()

print(f"Exact duplicate rows                    : {exact_duplicates:,}")
print(f"Duplicate spatial-temporal observations : {spatiotemporal_duplicates:,}")

print("\nUnique dates:")
print(jan_night["acq_date"].nunique())

print("\nDetections per day:")
print(
    jan_night
    .groupby("acq_date")
    .size()
    .describe()
)

Duplicate checks
Exact duplicate rows                    : 0
Duplicate spatial-temporal observations : 0

Unique dates:
31

Detections per day:
count    31.0000
mean    439.0000
std     128.1314
min     217.0000
25%     356.5000
50%     428.0000
75%     500.0000
max     825.0000
dtype: float64


In [7]:
# ---------------------------------------------------------
# DAILY SPATIAL OBJECT FORMATION
# ---------------------------------------------------------
# Each day's detections are grouped independently.
# This prevents temporal information from influencing
# the initial spatial object formation.
#
# We will test multiple spatial association radii.
# ---------------------------------------------------------

def create_daily_spatial_objects(df, radius_km):
    """
    Create spatial objects independently for each acquisition day.

    Parameters
    ----------
    df : DataFrame
        Nighttime VIIRS detections.
    radius_km : float
        Spatial association radius in km.

    Returns
    -------
    DataFrame
        One row per daily spatial object.
    """

    start = time.time()

    work = df.copy()

    coords_rad = np.radians(
        work[["latitude", "longitude"]].values
    )

    eps_rad = radius_km / EARTH_RADIUS_KM

    object_records = []
    object_id = 0

    for date, day_data in work.groupby("acq_date", sort=True):

        day_indices = day_data.index.to_numpy()

        day_coords = np.radians(
            day_data[["latitude", "longitude"]].values
        )

        model = DBSCAN(
            eps=eps_rad,
            min_samples=1,
            metric="haversine",
            algorithm="ball_tree"
        )

        labels = model.fit_predict(day_coords)

        for label in np.unique(labels):

            detection_indices = day_indices[labels == label]

            object_records.append({
                "daily_object_id": object_id,
                "acq_date": date,
                "detection_count": len(detection_indices),
                "detection_indices": detection_indices.tolist()
            })

            object_id += 1

    objects = pd.DataFrame(object_records)

    elapsed = time.time() - start

    print(
        f"Radius {radius_km:.3f} km → "
        f"{len(objects):,} daily objects "
        f"({elapsed:.2f}s)"
    )

    return objects

In [8]:
# ---------------------------------------------------------
# TEST DAILY SPATIAL OBJECT FORMATION
# ---------------------------------------------------------

daily_objects_by_radius = {}

for radius in SPATIAL_RADII_KM:

    daily_objects_by_radius[radius] = create_daily_spatial_objects(
        jan_night,
        radius
    )

Radius 0.375 km → 11,116 daily objects (0.38s)
Radius 0.500 km → 8,573 daily objects (0.34s)
Radius 0.750 km → 7,326 daily objects (0.33s)
Radius 1.000 km → 6,888 daily objects (0.45s)


In [9]:
# ---------------------------------------------------------
# DAILY OBJECT SIZE DISTRIBUTION
# ---------------------------------------------------------

for radius, objects in daily_objects_by_radius.items():

    print("\n" + "=" * 60)
    print(f"SPATIAL RADIUS: {radius:.3f} km")
    print("=" * 60)

    print(
        objects["detection_count"]
        .describe(
            percentiles=[
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )

    print(
        "\nSingleton objects:",
        (objects["detection_count"] == 1).sum(),
        f"({(objects['detection_count'] == 1).mean()*100:.2f}%)"
    )


SPATIAL RADIUS: 0.375 km
count   11116.0000
mean        1.2243
std         0.8027
min         1.0000
50%         1.0000
75%         1.0000
90%         2.0000
95%         2.0000
99%         4.0000
max        32.0000
Name: detection_count, dtype: float64

Singleton objects: 9522 (85.66%)

SPATIAL RADIUS: 0.500 km
count   8573.0000
mean       1.5874
std        1.6880
min        1.0000
50%        1.0000
75%        2.0000
90%        3.0000
95%        4.0000
99%        7.0000
max       36.0000
Name: detection_count, dtype: float64

Singleton objects: 6064 (70.73%)

SPATIAL RADIUS: 0.750 km
count   7326.0000
mean       1.8576
std        2.2195
min        1.0000
50%        1.0000
75%        2.0000
90%        3.0000
95%        5.0000
99%       10.0000
max       40.0000
Name: detection_count, dtype: float64

Singleton objects: 4661 (63.62%)

SPATIAL RADIUS: 1.000 km
count   6888.0000
mean       1.9758
std        2.3976
min        1.0000
50%        1.0000
75%        2.0000
90%        4.0000
95% 

In [10]:
# ---------------------------------------------------------
# DAILY OBJECT SPATIAL COMPACTNESS
# ---------------------------------------------------------

def haversine_distance_km(lat1, lon1, lat2, lon2):
    """
    Haversine distance between two geographic coordinates.
    """
    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


def calculate_daily_object_geometry(df, daily_objects):
    """
    Calculate centroid, maximum detection distance from centroid,
    and pairwise spatial diameter for each daily object.
    """

    records = []

    for _, obj in daily_objects.iterrows():

        indices = obj["detection_indices"]

        points = df.loc[
            indices,
            ["latitude", "longitude"]
        ].to_numpy()

        centroid_lat = points[:, 0].mean()
        centroid_lon = points[:, 1].mean()

        # Distance of every detection from centroid
        centroid_distances = haversine_distance_km(
            points[:, 0],
            points[:, 1],
            centroid_lat,
            centroid_lon
        )

        # Pairwise diameter
        if len(points) > 1:

            coords_rad = np.radians(points)

            tree = BallTree(
                coords_rad,
                metric="haversine"
            )

            distance_matrix = tree.kernel_density(
                coords_rad,
                h=1.0
            ) if False else None

            # Explicit pairwise calculation
            max_pairwise = 0.0

            for i in range(len(points)):
                distances = haversine_distance_km(
                    points[i, 0],
                    points[i, 1],
                    points[:, 0],
                    points[:, 1]
                )

                max_pairwise = max(
                    max_pairwise,
                    distances.max()
                )

        else:
            max_pairwise = 0.0

        records.append({
            "daily_object_id": obj["daily_object_id"],
            "acq_date": obj["acq_date"],
            "detection_count": len(indices),
            "centroid_lat": centroid_lat,
            "centroid_lon": centroid_lon,
            "max_distance_from_centroid_km": centroid_distances.max(),
            "mean_distance_from_centroid_km": centroid_distances.mean(),
            "spatial_diameter_km": max_pairwise
        })

    return pd.DataFrame(records)

In [11]:
# ---------------------------------------------------------
# CALCULATE GEOMETRY FOR ALL RADII
# ---------------------------------------------------------

daily_geometry_by_radius = {}

for radius, objects in daily_objects_by_radius.items():

    start = time.time()

    geometry = calculate_daily_object_geometry(
        jan_night,
        objects
    )

    daily_geometry_by_radius[radius] = geometry

    print(
        f"Radius {radius:.3f} km → "
        f"{len(geometry):,} objects "
        f"({time.time() - start:.2f}s)"
    )

Radius 0.375 km → 11,116 objects (16.58s)
Radius 0.500 km → 8,573 objects (12.81s)
Radius 0.750 km → 7,326 objects (11.17s)
Radius 1.000 km → 6,888 objects (9.76s)


In [12]:
# ---------------------------------------------------------
# DAILY OBJECT COMPACTNESS SUMMARY
# ---------------------------------------------------------

for radius, geometry in daily_geometry_by_radius.items():

    print("\n" + "=" * 65)
    print(f"RADIUS: {radius:.3f} km")
    print("=" * 65)

    print("\nMaximum distance from centroid (km):")

    print(
        geometry["max_distance_from_centroid_km"]
        .describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    print("\nSpatial diameter (km):")

    print(
        geometry["spatial_diameter_km"]
        .describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


RADIUS: 0.375 km

Maximum distance from centroid (km):
count   11116.0000
mean        0.0258
std         0.0818
min         0.0000
50%         0.0000
75%         0.0000
90%         0.1047
95%         0.1861
99%         0.3740
max         1.4959
Name: max_distance_from_centroid_km, dtype: float64

Spatial diameter (km):
count   11116.0000
mean        0.0499
std         0.1567
min         0.0000
50%         0.0000
75%         0.0000
90%         0.2089
95%         0.3687
99%         0.7454
max         2.5818
Name: spatial_diameter_km, dtype: float64

RADIUS: 0.500 km

Maximum distance from centroid (km):
count   8573.0000
mean       0.0783
std        0.1648
min        0.0000
50%        0.0000
75%        0.1173
90%        0.2462
95%        0.3820
99%        0.7284
max        1.9436
Name: max_distance_from_centroid_km, dtype: float64

Spatial diameter (km):
count   8573.0000
mean       0.1492
std        0.3065
min        0.0000
50%        0.0000
75%        0.2337
90%        0.4819
95%     

In [13]:
# ---------------------------------------------------------
# COMPACTNESS VIOLATION ANALYSIS
# ---------------------------------------------------------

COMPACTNESS_THRESHOLDS_KM = [
    0.375,
    0.500,
    0.750,
    1.000,
    1.500,
    2.000
]

compactness_summary = []

for radius, geometry in daily_geometry_by_radius.items():

    total = len(geometry)

    row = {
        "dbscan_radius_km": radius,
        "daily_objects": total
    }

    for threshold in COMPACTNESS_THRESHOLDS_KM:

        violations = (
            geometry["spatial_diameter_km"] > threshold
        ).sum()

        row[f"diameter_gt_{threshold:.3f}km"] = violations
        row[f"pct_gt_{threshold:.3f}km"] = (
            violations / total * 100
        )

    compactness_summary.append(row)

compactness_summary = pd.DataFrame(compactness_summary)

compactness_summary

,dbscan_radius_km,daily_objects,diameter_gt_0.375km,pct_gt_0.375km,diameter_gt_0.500km,pct_gt_0.500km,diameter_gt_0.750km,pct_gt_0.750km,diameter_gt_1.000km,pct_gt_1.000km,diameter_gt_1.500km,pct_gt_1.500km,diameter_gt_2.000km,pct_gt_2.000km
0,0.3750,11116,389,3.4995,245,2.2040,89,0.8006,48,0.4318,10,0.0900,3,0.0270
1,0.5000,8573,1699,19.8180,797,9.2966,395,4.6075,177,2.0646,60,0.6999,23,0.2683
2,0.7500,7326,2067,28.2146,1393,19.0145,686,9.3639,407,5.5556,182,2.4843,73,0.9965
3,1.0000,6888,2091,30.3571,1518,22.0383,912,13.2404,601,8.7253,302,4.3844,122,1.7712


In [14]:
# ---------------------------------------------------------
# COMPACTNESS SUMMARY — EASY TO READ
# ---------------------------------------------------------

display(
    compactness_summary[
        [
            "dbscan_radius_km",
            "daily_objects",
            "pct_gt_0.375km",
            "pct_gt_0.500km",
            "pct_gt_0.750km",
            "pct_gt_1.000km",
            "pct_gt_1.500km",
            "pct_gt_2.000km"
        ]
    ].round(2)
)

,dbscan_radius_km,daily_objects,pct_gt_0.375km,pct_gt_0.500km,pct_gt_0.750km,pct_gt_1.000km,pct_gt_1.500km,pct_gt_2.000km
0,0.3800,11116,3.5000,2.2000,0.8000,0.4300,0.0900,0.0300
1,0.5000,8573,19.8200,9.3000,4.6100,2.0600,0.7000,0.2700
2,0.7500,7326,28.2100,19.0100,9.3600,5.5600,2.4800,1.0000
3,1.0000,6888,30.3600,22.0400,13.2400,8.7300,4.3800,1.7700


In [15]:
# ---------------------------------------------------------
# WORST DAILY OBJECTS
# ---------------------------------------------------------

for radius, geometry in daily_geometry_by_radius.items():

    print("\n" + "=" * 70)
    print(f"RADIUS: {radius:.3f} km — LARGEST DAILY OBJECTS")
    print("=" * 70)

    display(
        geometry
        .sort_values(
            "spatial_diameter_km",
            ascending=False
        )
        .head(10)[
            [
                "daily_object_id",
                "acq_date",
                "detection_count",
                "max_distance_from_centroid_km",
                "mean_distance_from_centroid_km",
                "spatial_diameter_km"
            ]
        ]
    )


RADIUS: 0.375 km — LARGEST DAILY OBJECTS


,daily_object_id,acq_date,detection_count,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km
2202,2202,2024-01-07,32,1.4959,0.7701,2.5818
4604,4604,2024-01-13,26,1.3345,0.7555,2.5473
7471,7471,2024-01-21,7,1.1205,0.6403,2.2409
4436,4436,2024-01-12,20,1.1755,0.6142,1.9624
16,16,2024-01-01,10,0.9697,0.5339,1.9359
8759,8759,2024-01-25,9,0.9865,0.5296,1.8831
9688,9688,2024-01-27,9,0.8854,0.4877,1.7513
500,500,2024-01-02,17,0.8172,0.5239,1.5757
2294,2294,2024-01-07,8,0.7779,0.4264,1.5177
8851,8851,2024-01-25,12,0.7688,0.4642,1.5072



RADIUS: 0.500 km — LARGEST DAILY OBJECTS


,daily_object_id,acq_date,detection_count,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km
4451,4451,2024-01-15,24,1.9162,0.9730,3.4460
2557,2557,2024-01-10,24,1.7909,0.9641,3.4023
5750,5750,2024-01-21,29,1.8075,0.9346,3.3702
2867,2867,2024-01-11,34,1.8531,1.0015,3.3685
7392,7392,2024-01-27,36,1.9436,0.9537,3.3667
1990,1990,2024-01-08,29,1.9179,0.9131,3.2747
4782,4782,2024-01-16,32,1.7796,0.9475,3.2246
8343,8343,2024-01-31,20,1.6669,0.9382,3.2083
15,15,2024-01-01,33,1.6991,0.8433,3.2068
6776,6776,2024-01-25,32,1.7779,0.8782,3.1032



RADIUS: 0.750 km — LARGEST DAILY OBJECTS


,daily_object_id,acq_date,detection_count,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km
4887,4887,2024-01-21,31,1.8587,0.9777,3.5599
4515,4515,2024-01-19,20,1.9540,0.9904,3.5384
2711,2711,2024-01-12,24,2.1215,0.9962,3.4946
3757,3757,2024-01-15,24,1.9162,0.9730,3.4460
6649,6649,2024-01-29,19,1.7894,1.0794,3.4067
2162,2162,2024-01-10,28,2.1257,1.0712,3.4023
2423,2423,2024-01-11,37,1.9879,1.0541,3.3685
6322,6322,2024-01-27,36,1.9436,0.9537,3.3667
3461,3461,2024-01-14,30,1.8191,0.9488,3.3666
5839,5839,2024-01-25,18,2.1547,0.9251,3.3184



RADIUS: 1.000 km — LARGEST DAILY OBJECTS


,daily_object_id,acq_date,detection_count,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km
1577,1577,2024-01-08,32,2.7020,1.0533,4.1678
972,972,2024-01-05,23,2.0011,0.9584,3.9417
4601,4601,2024-01-21,32,1.9575,1.0095,3.5599
4246,4246,2024-01-19,20,1.9540,0.9904,3.5384
4588,4588,2024-01-21,11,2.3333,1.0864,3.5364
2546,2546,2024-01-12,24,2.1215,0.9962,3.4946
3530,3530,2024-01-15,25,2.0585,1.0187,3.4460
6249,6249,2024-01-29,19,1.7894,1.0794,3.4067
2038,2038,2024-01-10,28,2.1257,1.0712,3.4023
2281,2281,2024-01-11,37,1.9879,1.0541,3.3685


In [16]:
# ---------------------------------------------------------
# INSPECT 375m CHAINING CASES
# ---------------------------------------------------------

geometry_375 = daily_geometry_by_radius[0.375].copy()

problem_objects_375 = (
    geometry_375[
        geometry_375["spatial_diameter_km"] > 0.375
    ]
    .sort_values(
        "spatial_diameter_km",
        ascending=False
    )
    .reset_index(drop=True)
)

print("375m spatial-object chaining diagnostics")
print("=" * 60)

print(f"Total daily objects              : {len(geometry_375):,}")
print(f"Objects > 375m diameter          : {len(problem_objects_375):,}")
print(
    f"Percentage problematic           : "
    f"{len(problem_objects_375) / len(geometry_375) * 100:.2f}%"
)

print("\nDiameter distribution of problematic objects:")
print(
    problem_objects_375[
        "spatial_diameter_km"
    ].describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

print("\nLargest problematic objects:")
display(
    problem_objects_375.head(30)
)

375m spatial-object chaining diagnostics
Total daily objects              : 11,116
Objects > 375m diameter          : 389
Percentage problematic           : 3.50%

Diameter distribution of problematic objects:
count   389.0000
mean      0.6767
std       0.3205
min       0.3750
50%       0.6065
75%       0.7473
90%       1.0498
95%       1.2327
99%       1.9390
max       2.5818
Name: spatial_diameter_km, dtype: float64

Largest problematic objects:


,daily_object_id,acq_date,detection_count,centroid_lat,centroid_lon,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km
0,2202,2024-01-07,32,23.7711,86.3926,1.4959,0.7701,2.5818
1,4604,2024-01-13,26,23.7732,86.3940,1.3345,0.7555,2.5473
2,7471,2024-01-21,7,23.7743,86.3915,1.1205,0.6403,2.2409
3,4436,2024-01-12,20,31.3371,77.0663,1.1755,0.6142,1.9624
4,16,2024-01-01,10,23.7676,86.3932,0.9697,0.5339,1.9359
5,8759,2024-01-25,9,23.7682,86.3933,0.9865,0.5296,1.8831
6,9688,2024-01-27,9,23.7675,86.3937,0.8854,0.4877,1.7513
7,500,2024-01-02,17,20.9662,85.1736,0.8172,0.5239,1.5757
8,2294,2024-01-07,8,30.5099,79.6278,0.7779,0.4264,1.5177
9,8851,2024-01-25,12,20.9657,85.1734,0.7688,0.4642,1.5072


In [17]:
# ---------------------------------------------------------
# CHAINING RATIO
# ---------------------------------------------------------
# A compact object should have its detections relatively
# close to its centroid.
#
# A chain-like object tends to have:
#   large diameter
#   large centroid radius
#   relatively large mean distance
# ---------------------------------------------------------

problem_objects_375["chain_ratio"] = (
    problem_objects_375["spatial_diameter_km"]
    /
    problem_objects_375["max_distance_from_centroid_km"]
    .replace(0, np.nan)
)

display(
    problem_objects_375[
        [
            "daily_object_id",
            "acq_date",
            "detection_count",
            "max_distance_from_centroid_km",
            "mean_distance_from_centroid_km",
            "spatial_diameter_km",
            "chain_ratio"
        ]
    ]
    .head(30)
)

,daily_object_id,acq_date,detection_count,max_distance_from_centroid_km,mean_distance_from_centroid_km,spatial_diameter_km,chain_ratio
0,2202,2024-01-07,32,1.4959,0.7701,2.5818,1.7259
1,4604,2024-01-13,26,1.3345,0.7555,2.5473,1.9087
2,7471,2024-01-21,7,1.1205,0.6403,2.2409,1.9999
3,4436,2024-01-12,20,1.1755,0.6142,1.9624,1.6695
4,16,2024-01-01,10,0.9697,0.5339,1.9359,1.9964
5,8759,2024-01-25,9,0.9865,0.5296,1.8831,1.9088
6,9688,2024-01-27,9,0.8854,0.4877,1.7513,1.9781
7,500,2024-01-02,17,0.8172,0.5239,1.5757,1.9282
8,2294,2024-01-07,8,0.7779,0.4264,1.5177,1.9511
9,8851,2024-01-25,12,0.7688,0.4642,1.5072,1.9605


In [18]:
# ---------------------------------------------------------
# COMPLETE-LINKAGE SPATIAL GROUPING
# ---------------------------------------------------------
# Unlike DBSCAN, complete linkage prevents chain growth.
#
# A cluster can only be formed when the maximum pairwise
# distance within that cluster remains <= the threshold.
#
# Singleton detections are retained automatically.
# ---------------------------------------------------------

from sklearn.cluster import AgglomerativeClustering


def create_complete_linkage_objects(df, radius_km):

    start = time.time()

    object_records = []
    object_id = 0

    for date, day_data in df.groupby("acq_date", sort=True):

        day_indices = day_data.index.to_numpy()

        coords_rad = np.radians(
            day_data[["latitude", "longitude"]].values
        )

        # Pairwise haversine distance matrix
        lat = coords_rad[:, 0][:, None]
        lon = coords_rad[:, 1][:, None]

        dlat = lat - lat.T
        dlon = lon - lon.T

        a = (
            np.sin(dlat / 2) ** 2
            +
            np.cos(lat)
            * np.cos(lat.T)
            * np.sin(dlon / 2) ** 2
        )

        distance_matrix_km = (
            2
            * EARTH_RADIUS_KM
            * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        )

        model = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=radius_km,
            metric="precomputed",
            linkage="complete"
        )

        labels = model.fit_predict(
            distance_matrix_km
        )

        for label in np.unique(labels):

            detection_indices = day_indices[
                labels == label
            ]

            object_records.append({
                "daily_object_id": object_id,
                "acq_date": date,
                "detection_count": len(detection_indices),
                "detection_indices": detection_indices.tolist()
            })

            object_id += 1

    objects = pd.DataFrame(object_records)

    print(
        f"Complete-linkage {radius_km:.3f} km → "
        f"{len(objects):,} daily objects "
        f"({time.time() - start:.2f}s)"
    )

    return objects

In [19]:
# ---------------------------------------------------------
# COMPLETE-LINKAGE TEST
# ---------------------------------------------------------

start = time.time()

daily_objects_complete_375 = (
    create_complete_linkage_objects(
        jan_night,
        0.375
    )
)

print(
    f"\nTotal processing time: "
    f"{time.time() - start:.2f}s"
)

Complete-linkage 0.375 km → 11,678 daily objects (1.47s)

Total processing time: 1.47s


In [20]:
# ---------------------------------------------------------
# COMPARE DBSCAN vs COMPLETE LINKAGE
# ---------------------------------------------------------

print("DBSCAN 375m")
print("=" * 50)

dbscan_375 = daily_objects_by_radius[0.375]

print(
    dbscan_375["detection_count"].describe()
)

print(
    "Singleton %:",
    (
        dbscan_375["detection_count"].eq(1).mean()
        * 100
    )
)

print("\nComplete-linkage 375m")
print("=" * 50)

print(
    daily_objects_complete_375["detection_count"]
    .describe()
)

print(
    "Singleton %:",
    (
        daily_objects_complete_375["detection_count"].eq(1).mean()
        * 100
    )
)

DBSCAN 375m
count   11116.0000
mean        1.2243
std         0.8027
min         1.0000
25%         1.0000
50%         1.0000
75%         1.0000
max        32.0000
Name: detection_count, dtype: float64
Singleton %: 85.66030946383592

Complete-linkage 375m
count   11678.0000
mean        1.1654
std         0.4008
min         1.0000
25%         1.0000
50%         1.0000
75%         1.0000
max         4.0000
Name: detection_count, dtype: float64
Singleton %: 84.51789690015413


In [21]:
# ---------------------------------------------------------
# COMPLETE-LINKAGE GEOMETRY CHECK
# ---------------------------------------------------------

complete_geometry_375 = (
    calculate_daily_object_geometry(
        jan_night,
        daily_objects_complete_375
    )
)

print(
    complete_geometry_375[
        [
            "max_distance_from_centroid_km",
            "mean_distance_from_centroid_km",
            "spatial_diameter_km"
        ]
    ].describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

print("\nObjects exceeding 375m diameter:")

violations = (
    complete_geometry_375[
        complete_geometry_375["spatial_diameter_km"] > 0.375
    ]
)

print(
    f"{len(violations):,} / "
    f"{len(complete_geometry_375):,}"
)

print(
    f"Percentage: "
    f"{len(violations) / len(complete_geometry_375) * 100:.4f}%"
)

print("\nLargest diameter:")

print(
    complete_geometry_375[
        "spatial_diameter_km"
    ].max()
)

       max_distance_from_centroid_km  mean_distance_from_centroid_km  \
count                     11678.0000                      11678.0000   
mean                          0.0191                          0.0186   
std                           0.0485                          0.0473   
min                           0.0000                          0.0000   
50%                           0.0000                          0.0000   
75%                           0.0000                          0.0000   
90%                           0.1000                          0.0997   
95%                           0.1551                          0.1485   
99%                           0.1867                          0.1866   
max                           0.2384                          0.2022   

       spatial_diameter_km  
count           11678.0000  
mean                0.0377  
std                 0.0956  
min                 0.0000  
50%                 0.0000  
75%                 0.0000  
90% 

In [22]:
print(daily_objects_complete_375.columns.tolist())
print(daily_objects_complete_375.head())

['daily_object_id', 'acq_date', 'detection_count', 'detection_indices']
   daily_object_id   acq_date  detection_count detection_indices
0                0 2024-01-01                2        [116, 117]
1                1 2024-01-01                2        [127, 128]
2                2 2024-01-01                2        [199, 203]
3                3 2024-01-01                2        [129, 130]
4                4 2024-01-01                2        [222, 223]


In [23]:
# ---------------------------------------------------------
# TEMPORAL LINKING — COMPLETE-LINKAGE 375m
# ---------------------------------------------------------

def temporal_link_complete_linkage(
    detections,
    daily_objects,
    spatial_radius_km=0.375,
    temporal_gap_days=3
):
    objects = daily_objects.copy()
    objects["acq_date"] = pd.to_datetime(objects["acq_date"])

    # Calculate centroid for every daily object
    centroids = []

    for _, row in objects.iterrows():
        idx = row["detection_indices"]

        pts = detections.loc[idx, ["latitude", "longitude"]]

        centroids.append([
            pts["latitude"].mean(),
            pts["longitude"].mean()
        ])

    objects["centroid_lat"] = [x[0] for x in centroids]
    objects["centroid_lon"] = [x[1] for x in centroids]

    # -----------------------------------------------------
    # Spatial index
    # -----------------------------------------------------

    coords = np.radians(
        objects[["centroid_lat", "centroid_lon"]].values
    )

    tree = BallTree(coords, metric="haversine")

    radius_rad = spatial_radius_km / EARTH_RADIUS_KM

    n = len(objects)
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra != rb:
            parent[rb] = ra

    # -----------------------------------------------------
    # Temporal linking
    # -----------------------------------------------------

    for i in range(n):

        neighbours = tree.query_radius(
            coords[i:i+1],
            r=radius_rad
        )[0]

        date_i = objects.iloc[i]["acq_date"]

        for j in neighbours:

            if j <= i:
                continue

            date_j = objects.iloc[j]["acq_date"]

            gap = abs((date_j - date_i).days)

            if gap <= temporal_gap_days:
                union(i, j)

    # -----------------------------------------------------
    # Assign event IDs
    # -----------------------------------------------------

    roots = np.array([find(i) for i in range(n)])

    root_to_event = {}
    event_ids = []

    next_event_id = 0

    for root in roots:

        if root not in root_to_event:
            root_to_event[root] = next_event_id
            next_event_id += 1

        event_ids.append(root_to_event[root])

    objects["event_id"] = event_ids

    return objects


# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------

complete_events_375 = temporal_link_complete_linkage(
    jan_night,
    daily_objects_complete_375,
    spatial_radius_km=0.375,
    temporal_gap_days=3
)

print(
    f"Daily objects: "
    f"{len(daily_objects_complete_375):,}"
)

print(
    f"Final events: "
    f"{complete_events_375['event_id'].nunique():,}"
)

Daily objects: 11,678
Final events: 4,893


In [24]:
# ---------------------------------------------------------
# FINAL EVENT SPATIAL COHERENCE CHECK
# ---------------------------------------------------------

event_stats = []

for event_id, obj_group in complete_events_375.groupby("event_id"):

    all_indices = []

    for indices in obj_group["detection_indices"]:
        all_indices.extend(indices)

    pts = jan_night.loc[
        all_indices,
        ["latitude", "longitude"]
    ].values

    # Haversine pairwise distances
    rad = np.radians(pts)

    lat1 = rad[:, None, 0]
    lat2 = rad[None, :, 0]

    dlat = lat2 - lat1
    dlon = rad[None, :, 1] - rad[:, None, 1]

    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2)**2
    )

    distances = (
        2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        * EARTH_RADIUS_KM
    )

    event_stats.append({
        "event_id": event_id,
        "detection_count": len(all_indices),
        "daily_object_count": len(obj_group),
        "active_days": obj_group["acq_date"].nunique(),
        "spatial_diameter_km": distances.max()
    })

event_stats = pd.DataFrame(event_stats)

print(event_stats.describe(
    percentiles=[.50, .75, .90, .95, .99]
))

print("\nEvents exceeding 375m:")
violations = event_stats[
    event_stats["spatial_diameter_km"] > 0.375
]

print(
    f"{len(violations):,} / "
    f"{len(event_stats):,}"
)

print(
    f"{len(violations) / len(event_stats) * 100:.4f}%"
)

print(
    "\nLargest event diameter:",
    event_stats["spatial_diameter_km"].max()
)

       event_id  detection_count  daily_object_count  active_days  \
count 4893.0000        4893.0000           4893.0000    4893.0000   
mean  2446.0000           2.7813              2.3867       1.7370   
std   1412.6318          13.1317             10.1482       2.8661   
min      0.0000           1.0000              1.0000       1.0000   
50%   2446.0000           1.0000              1.0000       1.0000   
75%   3669.0000           2.0000              1.0000       1.0000   
90%   4402.8000           3.0000              3.0000       2.0000   
95%   4647.4000           6.4000              6.0000       5.0000   
99%   4843.0800          38.0800             32.0000      17.0000   
max   4892.0000         694.0000            515.0000      31.0000   

       spatial_diameter_km  
count            4893.0000  
mean                0.1008  
std                 0.2457  
min                 0.0000  
50%                 0.0000  
75%                 0.0600  
90%                 0.3637  
95%     

In [25]:
# ---------------------------------------------------------
# INSPECT SPATIALLY EXPANDED EVENTS
# ---------------------------------------------------------

expanded = event_stats[
    event_stats["spatial_diameter_km"] > 0.375
].copy()

print("Expanded events:", len(expanded))

print("\nBy active days:")
print(
    expanded["active_days"].describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

print("\nDiameter distribution:")
print(
    expanded["spatial_diameter_km"].describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

print("\nLargest events:")
print(
    expanded.sort_values(
        "spatial_diameter_km",
        ascending=False
    ).head(20)
)

Expanded events: 402

By active days:
count   402.0000
mean      8.3333
std       6.9993
min       1.0000
50%       6.0000
75%      11.7500
90%      20.0000
95%      25.0000
99%      27.0000
max      31.0000
Name: active_days, dtype: float64

Diameter distribution:
count   402.0000
mean      0.7215
std       0.4442
min       0.3753
50%       0.5725
75%       0.8003
90%       1.2048
95%       1.7057
99%       2.4313
max       3.7180
Name: spatial_diameter_km, dtype: float64

Largest events:
      event_id  detection_count  daily_object_count  active_days  \
17          17              694                 515           27   
48          48              205                 157           31   
127        127               86                  75           26   
80          80              125                 100           19   
135        135              177                 135           16   
19          19               97                  74           17   
25          25               

In [26]:
# ---------------------------------------------------------
# EXPORT V11 EVENT DATASET
# ---------------------------------------------------------

complete_events_375.to_pickle(
    "v11_complete_events_375.pkl"
)

print("Saved:")
print("v11_complete_events_375.pkl")

Saved:
v11_complete_events_375.pkl


In [27]:
event_stats.to_csv(
    "v11_event_spatial_quality.csv",
    index=False
)

print("Saved:")
print("v11_event_spatial_quality.csv")

Saved:
v11_event_spatial_quality.csv


In [28]:
# ---------------------------------------------------------
# V11 — FINAL EVENT DATASET QUALITY CHECK
# ---------------------------------------------------------

events_v11 = event_stats.copy()

print("Shape:", events_v11.shape)

print("\nColumns:")
print(events_v11.columns.tolist())

print("\nMissing values:")
print(events_v11.isna().sum())

print("\nDuplicate event IDs:")
print(events_v11["event_id"].duplicated().sum())

print("\nEvent count:")
print(events_v11["event_id"].nunique())

print("\nDetection count:")
print(events_v11["detection_count"].describe())

print("\nActive days:")
print(events_v11["active_days"].describe())

print("\nSpatial diameter:")
print(events_v11["spatial_diameter_km"].describe())

print("\nInvalid values:")

print(
    "detection_count <= 0:",
    (events_v11["detection_count"] <= 0).sum()
)

print(
    "daily_object_count <= 0:",
    (events_v11["daily_object_count"] <= 0).sum()
)

print(
    "active_days <= 0:",
    (events_v11["active_days"] <= 0).sum()
)

print(
    "spatial_diameter < 0:",
    (events_v11["spatial_diameter_km"] < 0).sum()
)

print("\nDone.")

Shape: (4893, 5)

Columns:
['event_id', 'detection_count', 'daily_object_count', 'active_days', 'spatial_diameter_km']

Missing values:
event_id               0
detection_count        0
daily_object_count     0
active_days            0
spatial_diameter_km    0
dtype: int64

Duplicate event IDs:
0

Event count:
4893

Detection count:
count   4893.0000
mean       2.7813
std       13.1317
min        1.0000
25%        1.0000
50%        1.0000
75%        2.0000
max      694.0000
Name: detection_count, dtype: float64

Active days:
count   4893.0000
mean       1.7370
std        2.8661
min        1.0000
25%        1.0000
50%        1.0000
75%        1.0000
max       31.0000
Name: active_days, dtype: float64

Spatial diameter:
count   4893.0000
mean       0.1008
std        0.2457
min        0.0000
25%        0.0000
50%        0.0000
75%        0.0600
max        3.7180
Name: spatial_diameter_km, dtype: float64

Invalid values:
detection_count <= 0: 0
daily_object_count <= 0: 0
active_days <= 0: 

In [60]:
# ============================================================
# V11 DETECTION → DAILY OBJECT → EVENT MAPPING
# ============================================================

detection_event_map = []

for _, row in complete_events_375.iterrows():

    for idx in row["detection_indices"]:

        detection_event_map.append({
            "detection_index": idx,
            "daily_object_id": row["daily_object_id"],
            "event_id": row["event_id"],
            "acq_date": row["acq_date"],
            "latitude": jan_night.loc[idx, "latitude"],
            "longitude": jan_night.loc[idx, "longitude"]
        })

detection_event_map = pd.DataFrame(
    detection_event_map
)

print("V11 detection → daily object → event mapping")
print("=" * 55)

print("Mapped detections     :", len(detection_event_map))
print(
    "Unique detections     :",
    detection_event_map["detection_index"].nunique()
)
print(
    "Unique daily objects  :",
    detection_event_map["daily_object_id"].nunique()
)
print(
    "Unique events         :",
    detection_event_map["event_id"].nunique()
)

print(
    "Duplicate detection assignments:",
    detection_event_map[
        "detection_index"
    ].duplicated().sum()
)

V11_MAPPING_FILE = (
    "viirs_v11_detection_event_mapping.csv"
)

detection_event_map.to_csv(
    V11_MAPPING_FILE,
    index=False
)

print(f"\nSaved: {V11_MAPPING_FILE}")

V11 detection → daily object → event mapping
Mapped detections     : 13609
Unique detections     : 13609
Unique daily objects  : 11678
Unique events         : 4893
Duplicate detection assignments: 0

Saved: viirs_v11_detection_event_mapping.csv


In [61]:
print(detection_event_map.columns.tolist())

['detection_index', 'daily_object_id', 'event_id', 'acq_date', 'latitude', 'longitude']


In [62]:
V11_MAPPING_FILE = "viirs_v11_detection_event_mapping.csv"

detection_event_map.to_csv(
    V11_MAPPING_FILE,
    index=False
)

print(f"Saved: {V11_MAPPING_FILE}")

Saved: viirs_v11_detection_event_mapping.csv


In [59]:
# ---------------------------------------------------------
# SAVE V11 TRACEABILITY MAPPING
# ---------------------------------------------------------

V11_MAPPING_FILE = (
    "viirs_v11_detection_event_mapping.csv"
)

detection_event_map.to_csv(
    V11_MAPPING_FILE,
    index=False
)

print(
    f"\nSaved V11 mapping: {V11_MAPPING_FILE}"
)


Saved V11 mapping: viirs_v11_detection_event_mapping.csv


In [30]:
# ---------------------------------------------------------
# SAVE FROZEN V11 EVENT DATASET
# ---------------------------------------------------------

V11_FINAL_EVENT_FILE = "viirs_v11_final_event_dataset.csv"

events_v11.to_csv(
    V11_FINAL_EVENT_FILE,
    index=False
)

print(f"Saved: {V11_FINAL_EVENT_FILE}")
print(f"Shape: {events_v11.shape}")

Saved: viirs_v11_final_event_dataset.csv
Shape: (4893, 24)


In [31]:
# ---------------------------------------------------------
# TASK 3 — LOAD EXISTING GEOGRAPHIC CONTEXT
# ---------------------------------------------------------

events_v11 = pd.read_csv("viirs_v11_final_event_dataset.csv")

osm_points = pd.read_csv("osm_points_v7.csv")

print("Events:", events_v11.shape)
print("OSM:", osm_points.shape)

print("\nEvent columns:")
print(events_v11.columns.tolist())

print("\nOSM columns:")
print(osm_points.columns.tolist())

Events: (4893, 24)
OSM: (1981, 15)

Event columns:
['event_id', 'mean_frp', 'max_frp', 'std_frp', 'mean_bright_ti4', 'max_bright_ti4', 'std_bright_ti4', 'mean_bright_ti5', 'max_bright_ti5', 'std_bright_ti5', 'start_date', 'end_date', 'active_days', 'detection_count', 'duration_days', 'activity_frequency', 'detections_per_active_day', 'centroid_lat', 'centroid_lon', 'daily_object_count', 'spatial_diameter_km', 'frp_range', 'ti4_range', 'ti5_range']

OSM columns:
['osm_type', 'osm_id', 'latitude', 'longitude', 'industrial', 'landuse', 'power', 'man_made', 'building', 'product', 'plant_source', 'plant_method', 'resource', 'description', 'source_file']


In [32]:
# ---------------------------------------------------------
# TASK 3A — NORMALIZE OSM INTO DOMAIN ENTITY TYPES
# ---------------------------------------------------------

osm = osm_points.copy()

# Normalize text fields
for col in [
    "industrial",
    "landuse",
    "power",
    "man_made",
    "building",
    "product",
    "plant_source",
    "plant_method",
    "resource",
    "description"
]:
    osm[col] = (
        osm[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ---------------------------------------------------------
# DOMAIN ENTITY FLAGS
# ---------------------------------------------------------

osm["entity_industrial_zone"] = (
    osm["landuse"] == "industrial"
)

osm["entity_factory"] = osm["industrial"].isin([
    "factory",
    "concrete_plant",
    "machine_shop",
    "food_industry",
    "biotechnology company",
    "biotechnology_company",
    "pharmaceutical company",
    "pharmaceutical_company",
    "research company",
    "research institute",
    "laboratory",
    "agrochemical company",
    "refractory_supplier",
    "oil",
    "oil_mill",
    "rice_mill",
    "grinding_mill",
    "sawmill",
    "mill"
])

osm["entity_mine"] = (
    (osm["industrial"] == "mine") |
    (osm["landuse"] == "quarry")
)

osm["entity_brick"] = osm["industrial"].isin([
    "brickyard",
    "brickworks"
]) | (osm["man_made"] == "kiln")

osm["entity_works"] = (
    osm["man_made"] == "works"
)

osm["entity_depot"] = osm["industrial"].isin([
    "depot",
    "bus_depot"
])

osm["entity_power"] = (
    osm["power"] == "plant"
)

osm["entity_other_industry"] = osm["industrial"].isin([
    "slaughterhouse",
    "scrap_yard",
    "warehouse",
    "port",
    "cooling",
    "distributor",
    "business"
])

entity_cols = [
    "entity_industrial_zone",
    "entity_factory",
    "entity_mine",
    "entity_brick",
    "entity_works",
    "entity_depot",
    "entity_power",
    "entity_other_industry"
]

osm["entity_any_industry"] = osm[entity_cols].any(axis=1)

print("OSM entity counts:")
print(
    osm[entity_cols]
    .sum()
    .sort_values(ascending=False)
)

print("\nTotal OSM points:", len(osm))
print(
    "Industrial-context points:",
    osm["entity_any_industry"].sum()
)

OSM entity counts:
entity_works              1563
entity_industrial_zone     360
entity_other_industry      171
entity_factory              67
entity_depot                39
entity_brick                 6
entity_mine                  1
entity_power                 0
dtype: int64

Total OSM points: 1981
Industrial-context points: 1971


In [33]:
# ---------------------------------------------------------
# TASK 3B — OSM PROXIMITY FEATURES
# ---------------------------------------------------------

from sklearn.neighbors import BallTree

# Event coordinates
event_coords = np.radians(
    events_v11[["centroid_lat", "centroid_lon"]].values
)

# OSM coordinates
osm_coords = np.radians(
    osm[["latitude", "longitude"]].values
)

# Entity-specific trees
entity_trees = {}

for entity in entity_cols:
    mask = osm[entity].values

    if mask.sum() > 0:
        entity_trees[entity] = BallTree(
            osm_coords[mask],
            metric="haversine"
        )

# ---------------------------------------------------------
# DISTANCES + COUNTS
# ---------------------------------------------------------

radii_km = {
    "375m": 0.375,
    "1km": 1.0,
    "3km": 3.0
}

osm_features = pd.DataFrame({
    "event_id": events_v11["event_id"]
})

for entity in entity_cols:

    prefix = entity.replace("entity_", "")

    if entity not in entity_trees:

        osm_features[f"nearest_{prefix}_km"] = np.nan

        for radius_name in radii_km:
            osm_features[
                f"{prefix}_count_{radius_name}"
            ] = 0

        continue

    mask = osm[entity].values
    entity_coords = osm_coords[mask]

    tree = entity_trees[entity]

    # Nearest distance
    dist, _ = tree.query(
        event_coords,
        k=1
    )

    osm_features[
        f"nearest_{prefix}_km"
    ] = dist[:, 0] * EARTH_RADIUS_KM

    # Radius counts
    for radius_name, radius_km in radii_km.items():

        counts = tree.query_radius(
            event_coords,
            r=radius_km / EARTH_RADIUS_KM,
            count_only=True
        )

        osm_features[
            f"{prefix}_count_{radius_name}"
        ] = counts

# ---------------------------------------------------------
# MERGE
# ---------------------------------------------------------

events_geo = events_v11.merge(
    osm_features,
    on="event_id",
    how="left"
)

print("Shape:", events_geo.shape)

print("\nNew OSM features:")
print(
    [
        c for c in events_geo.columns
        if c not in events_v11.columns
    ]
)

Shape: (4893, 56)

New OSM features:
['nearest_industrial_zone_km', 'industrial_zone_count_375m', 'industrial_zone_count_1km', 'industrial_zone_count_3km', 'nearest_factory_km', 'factory_count_375m', 'factory_count_1km', 'factory_count_3km', 'nearest_mine_km', 'mine_count_375m', 'mine_count_1km', 'mine_count_3km', 'nearest_brick_km', 'brick_count_375m', 'brick_count_1km', 'brick_count_3km', 'nearest_works_km', 'works_count_375m', 'works_count_1km', 'works_count_3km', 'nearest_depot_km', 'depot_count_375m', 'depot_count_1km', 'depot_count_3km', 'nearest_power_km', 'power_count_375m', 'power_count_1km', 'power_count_3km', 'nearest_other_industry_km', 'other_industry_count_375m', 'other_industry_count_1km', 'other_industry_count_3km']


In [34]:
# ---------------------------------------------------------
# OSM COVERAGE DIAGNOSTIC
# ---------------------------------------------------------

for radius_name in ["375m", "1km", "3km"]:

    print(f"\n===== {radius_name} =====")

    for entity in entity_cols:

        prefix = entity.replace("entity_", "")

        col = f"{prefix}_count_{radius_name}"

        n = (events_geo[col] > 0).sum()

        print(
            f"{prefix:20s}: "
            f"{n:4d} events "
            f"({n / len(events_geo) * 100:.2f}%)"
        )


===== 375m =====
industrial_zone     :    0 events (0.00%)
factory             :    0 events (0.00%)
mine                :    0 events (0.00%)
brick               :    0 events (0.00%)
works               :    6 events (0.12%)
depot               :    0 events (0.00%)
power               :    0 events (0.00%)
other_industry      :    0 events (0.00%)

===== 1km =====
industrial_zone     :    0 events (0.00%)
factory             :    0 events (0.00%)
mine                :    0 events (0.00%)
brick               :    0 events (0.00%)
works               :   21 events (0.43%)
depot               :    0 events (0.00%)
power               :    0 events (0.00%)
other_industry      :    0 events (0.00%)

===== 3km =====
industrial_zone     :    3 events (0.06%)
factory             :    0 events (0.00%)
mine                :    0 events (0.00%)
brick               :    0 events (0.00%)
works               :  100 events (2.04%)
depot               :    1 events (0.02%)
power               :   

In [35]:
# ---------------------------------------------------------
# TASK 3C — WORLDCOVER CENTROID CONTEXT
# ---------------------------------------------------------

import glob
import rasterio

WORLD_COVER_DIR = "worldcover_india"

worldcover_files = sorted(
    glob.glob(
        os.path.join(
            WORLD_COVER_DIR,
            "ESA_WorldCover_10m_2021_v200_*_Map.tif"
        )
    )
)

print("WorldCover files:", len(worldcover_files))

# Build tile index
tiles = []

for path in worldcover_files:

    try:
        with rasterio.open(path) as src:
            bounds = src.bounds

            tiles.append({
                "path": path,
                "min_lon": bounds.left,
                "max_lon": bounds.right,
                "min_lat": bounds.bottom,
                "max_lat": bounds.top
            })

    except Exception:
        pass

print("Valid tiles:", len(tiles))

# ---------------------------------------------------------
# ASSIGN EVENTS TO TILES
# ---------------------------------------------------------

wc_class = np.full(len(events_geo), np.nan)

for tile in tiles:

    mask = (
        (events_geo["centroid_lon"] >= tile["min_lon"]) &
        (events_geo["centroid_lon"] < tile["max_lon"]) &
        (events_geo["centroid_lat"] >= tile["min_lat"]) &
        (events_geo["centroid_lat"] < tile["max_lat"]) &
        np.isnan(wc_class)
    )

    indices = np.where(mask)[0]

    if len(indices) == 0:
        continue

    try:
        with rasterio.open(tile["path"]) as src:

            coords = [
                (
                    events_geo.iloc[i]["centroid_lon"],
                    events_geo.iloc[i]["centroid_lat"]
                )
                for i in indices
            ]

            values = list(src.sample(coords))

            for i, value in zip(indices, values):
                wc_class[i] = value[0]

    except Exception as e:
        print(
            "Tile failed:",
            os.path.basename(tile["path"])
        )

events_geo["worldcover_class"] = wc_class

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------

print("\nWorldCover coverage:")
print(
    events_geo["worldcover_class"]
    .notna()
    .sum(),
    "/",
    len(events_geo)
)

print("\nClass distribution:")

print(
    events_geo["worldcover_class"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nClass names:")

WORLDCOVER_CLASSES = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare/sparse vegetation",
    70: "Snow/ice",
    80: "Permanent water",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss/lichen"
}

print(
    events_geo["worldcover_class"]
    .map(WORLDCOVER_CLASSES)
    .value_counts(dropna=False)
)

WorldCover files: 102
Valid tiles: 102
Tile failed: ESA_WorldCover_10m_2021_v200_N33E072_Map.tif
Tile failed: ESA_WorldCover_10m_2021_v200_N33E075_Map.tif

WorldCover coverage:
4528 / 4893

Class distribution:
worldcover_class
10.0000     1418
20.0000      149
30.0000      890
40.0000      867
50.0000      632
60.0000      511
80.0000       56
90.0000        3
100.0000       2
NaN          365
Name: count, dtype: int64

Class names:
worldcover_class
Tree cover                1418
Grassland                  890
Cropland                   867
Built-up                   632
Bare/sparse vegetation     511
NaN                        365
Shrubland                  149
Permanent water             56
Herbaceous wetland           3
Moss/lichen                  2
Name: count, dtype: int64


In [36]:
# ---------------------------------------------------------
# TASK 3D — WORLDCOVER CONTEXT FEATURES
# ---------------------------------------------------------

WC_FEATURES = {
    10: "tree_cover",
    20: "shrubland",
    30: "grassland",
    40: "cropland",
    50: "built_up",
    60: "bare_sparse",
    70: "snow_ice",
    80: "permanent_water",
    90: "wetland",
    95: "mangroves",
    100: "moss_lichen"
}

# Create explicit missing indicator
events_geo["worldcover_missing"] = (
    events_geo["worldcover_class"].isna().astype(int)
)

# One-hot contextual indicators
for code, name in WC_FEATURES.items():
    events_geo[f"wc_{name}"] = (
        events_geo["worldcover_class"] == code
    ).astype(int)

print("WorldCover feature columns:")

print([
    c for c in events_geo.columns
    if c.startswith("wc_") or c == "worldcover_missing"
])

print("\nMissing WorldCover:")
print(
    events_geo["worldcover_missing"].sum(),
    "/",
    len(events_geo)
)

print("\nWorldCover feature totals:")

print(
    events_geo[
        [
            c for c in events_geo.columns
            if c.startswith("wc_")
        ]
    ].sum()
)

WorldCover feature columns:
['worldcover_missing', 'wc_tree_cover', 'wc_shrubland', 'wc_grassland', 'wc_cropland', 'wc_built_up', 'wc_bare_sparse', 'wc_snow_ice', 'wc_permanent_water', 'wc_wetland', 'wc_mangroves', 'wc_moss_lichen']

Missing WorldCover:
365 / 4893

WorldCover feature totals:
wc_tree_cover         1418
wc_shrubland           149
wc_grassland           890
wc_cropland            867
wc_built_up            632
wc_bare_sparse         511
wc_snow_ice              0
wc_permanent_water      56
wc_wetland               3
wc_mangroves             0
wc_moss_lichen           2
dtype: int64


In [37]:
# ---------------------------------------------------------
# TASK 4 — BUILD MODELLING DATASET
# ---------------------------------------------------------

# Core behavioural features
behavior_features = [
    "mean_frp",
    "max_frp",
    "std_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "active_days",
    "duration_days",
    "activity_frequency",
    "detections_per_active_day",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km",
    "frp_range",
    "ti4_range",
    "ti5_range"
]

# Geographic features
geo_features = [
    c for c in events_geo.columns
    if (
        c.startswith("nearest_")
        or "_count_" in c
        or c.startswith("wc_")
        or c == "worldcover_missing"
    )
]

model_features = behavior_features + geo_features

model_df = events_geo[
    ["event_id", "centroid_lat", "centroid_lon"]
    + model_features
].copy()

# Replace infinite values if any
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Events:", len(model_df))
print("Behaviour features:", len(behavior_features))
print("Geographic features:", len(geo_features))
print("Total modelling features:", len(model_features))

print("\nMissing values:")
print(
    model_df[model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nShape:", model_df.shape)

Events: 4893
Behaviour features: 19
Geographic features: 44
Total modelling features: 63

Missing values:
nearest_power_km              4893
max_frp                          0
mean_frp                         0
mean_bright_ti4                  0
max_bright_ti4                   0
std_bright_ti4                   0
std_frp                          0
max_bright_ti5                   0
std_bright_ti5                   0
active_days                      0
duration_days                    0
activity_frequency               0
detections_per_active_day        0
detection_count                  0
mean_bright_ti5                  0
spatial_diameter_km              0
frp_range                        0
ti4_range                        0
ti5_range                        0
nearest_industrial_zone_km       0
dtype: int64

Shape: (4893, 66)


In [38]:
# ---------------------------------------------------------
# TASK 4A — CLEAN MODELLING FEATURES
# ---------------------------------------------------------

# Remove completely unavailable OSM feature
model_df = model_df.drop(
    columns=["nearest_power_km"]
)

geo_features = [
    c for c in geo_features
    if c != "nearest_power_km"
]

# Separate WorldCover missingness from actual context
context_features = [
    c for c in geo_features
    if c != "worldcover_missing"
]

data_quality_features = [
    "worldcover_missing"
]

# Check remaining missing values
remaining_missing = (
    model_df[behavior_features + context_features]
    .isna()
    .sum()
)

print("Remaining missing values:")
print(
    remaining_missing[
        remaining_missing > 0
    ]
)

print("\nFinal feature counts:")
print("Behaviour:", len(behavior_features))
print("Geographic context:", len(context_features))
print("Data-quality:", len(data_quality_features))

print(
    "Total:",
    len(behavior_features)
    + len(context_features)
    + len(data_quality_features)
)

Remaining missing values:
Series([], dtype: int64)

Final feature counts:
Behaviour: 19
Geographic context: 42
Data-quality: 1
Total: 62


In [39]:
# ---------------------------------------------------------
# TASK 5 — MODELLING MATRICES
# ---------------------------------------------------------

# A: Behaviour only
X_behavior = model_df[behavior_features].copy()

# B: Behaviour + geographic context
X_combined = model_df[
    behavior_features + context_features
].copy()

# C: Geographic context only
X_geographic = model_df[context_features].copy()

print("Behaviour matrix:")
print(X_behavior.shape)

print("\nCombined matrix:")
print(X_combined.shape)

print("\nGeographic matrix:")
print(X_geographic.shape)

print("\nAll numeric:")
print(
    "Behaviour:",
    X_behavior.select_dtypes(include=np.number).shape
)

print(
    "Combined:",
    X_combined.select_dtypes(include=np.number).shape
)

print(
    "Geographic:",
    X_geographic.select_dtypes(include=np.number).shape
)

Behaviour matrix:
(4893, 19)

Combined matrix:
(4893, 61)

Geographic matrix:
(4893, 42)

All numeric:
Behaviour: (4893, 19)
Combined: (4893, 61)
Geographic: (4893, 42)


In [40]:
# ---------------------------------------------------------
# TASK 6 — K-MEANS BEHAVIOUR BASELINE
# ---------------------------------------------------------

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler_behavior = RobustScaler()

Xb_scaled = scaler_behavior.fit_transform(X_behavior)

results_behavior = []

for k in range(2, 7):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = km.fit_predict(Xb_scaled)

    silhouette = silhouette_score(
        Xb_scaled,
        labels
    )

    results_behavior.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette
    })

results_behavior = pd.DataFrame(results_behavior)

print(results_behavior)

   k       inertia  silhouette
0  2 20777586.7649      0.8461
1  3 14014385.1921      0.8131
2  4 11521489.9173      0.8014
3  5  9670463.5649      0.8187
4  6  8069328.8390      0.8026


In [41]:
# ---------------------------------------------------------
# TASK 6B — K=2 BEHAVIOUR CLUSTER PROFILE
# ---------------------------------------------------------

km_behavior = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

behavior_labels = km_behavior.fit_predict(Xb_scaled)

clustered_behavior = model_df[
    ["event_id"] + behavior_features
].copy()

clustered_behavior["cluster"] = behavior_labels

print("Cluster sizes:")
print(
    clustered_behavior["cluster"]
    .value_counts()
    .sort_index()
)

print("\nCluster profiles:")
print(
    clustered_behavior
    .groupby("cluster")[behavior_features]
    .mean()
    .T
)

Cluster sizes:
cluster
0    4500
1     393
Name: count, dtype: int64

Cluster profiles:
cluster                          0        1
mean_frp                    1.3105   1.5769
max_frp                     1.3698   2.7813
std_frp                     0.0701   0.6854
mean_bright_ti4           304.8679 306.2351
max_bright_ti4            305.4788 318.0152
std_bright_ti4              0.7264   7.4185
mean_bright_ti5           283.3105 285.8947
max_bright_ti5            283.4203 289.2445
std_bright_ti5              0.1296   2.9611
active_days                 1.1789   8.1272
duration_days               1.2593  10.2163
activity_frequency          0.9857   0.8095
detections_per_active_day   1.1256   1.9257
detection_count             1.3333  19.3613
daily_object_count          1.2018  15.9542
spatial_diameter_km         0.0519   0.6599
frp_range                   0.0594   1.2044
ti4_range                   0.6109  11.7801
ti5_range                   0.1098   3.3498


In [42]:
# ---------------------------------------------------------
# TASK 6C — PERSISTENCE PROFILE
# ---------------------------------------------------------

print("\nActive days by cluster:")
print(
    clustered_behavior
    .groupby("cluster")["active_days"]
    .describe(
        percentiles=[.50, .75, .90, .95]
    )
)

print("\nSpatial diameter by cluster:")
print(
    clustered_behavior
    .groupby("cluster")["spatial_diameter_km"]
    .describe(
        percentiles=[.50, .75, .90, .95]
    )
)


Active days by cluster:
            count   mean    std    min    50%     75%     90%     95%     max
cluster                                                                      
0       4500.0000 1.1789 0.7562 1.0000 1.0000  1.0000  1.0000  2.0000 13.0000
1        393.0000 8.1272 7.1720 1.0000 5.0000 12.0000 20.0000 25.0000 31.0000

Spatial diameter by cluster:
            count   mean    std    min    50%    75%    90%    95%    max
cluster                                                                  
0       4500.0000 0.0519 0.1240 0.0000 0.0000 0.0000 0.2621 0.3493 1.3263
1        393.0000 0.6599 0.4856 0.0198 0.5278 0.7870 1.2026 1.7234 3.7180


In [43]:
# ---------------------------------------------------------
# TASK 6D — CLUSTER / PERSISTENCE OVERLAP
# ---------------------------------------------------------

for days in [2, 3, 5, 7, 10, 15, 20]:

    persistent = (
        clustered_behavior["active_days"] >= days
    )

    print(
        f">= {days:2d} days | "
        f"All events: {persistent.sum():4d} | "
        f"Cluster 1: "
        f"{((clustered_behavior['cluster'] == 1) & persistent).sum():4d}"
    )

print("\nCluster 1 active-day distribution:")

print(
    clustered_behavior[
        clustered_behavior["cluster"] == 1
    ]["active_days"]
    .value_counts()
    .sort_index()
)

>=  2 days | All events:  762 | Cluster 1:  356
>=  3 days | All events:  470 | Cluster 1:  298
>=  5 days | All events:  282 | Cluster 1:  230
>=  7 days | All events:  192 | Cluster 1:  172
>= 10 days | All events:  123 | Cluster 1:  119
>= 15 days | All events:   73 | Cluster 1:   73
>= 20 days | All events:   42 | Cluster 1:   42

Cluster 1 active-day distribution:
active_days
1     37
2     58
3     39
4     29
5     35
6     23
7     19
8     17
9     17
10    11
11     9
12     7
13     7
14    12
15    12
16    10
17     5
19     4
20     6
21     1
22     3
23     6
24     4
25     8
26     6
27     5
28     1
29     1
31     1
Name: count, dtype: int64


In [44]:
# ---------------------------------------------------------
# TASK 7 — DBSCAN BEHAVIOUR COMPARISON
# ---------------------------------------------------------

from sklearn.cluster import DBSCAN

dbscan_results = []

for eps in [0.5, 0.75, 1.0, 1.25, 1.5]:

    db = DBSCAN(
        eps=eps,
        min_samples=10,
        metric="euclidean",
        n_jobs=-1
    )

    labels = db.fit_predict(Xb_scaled)

    n_clusters = len(
        set(labels) - {-1}
    )

    n_noise = (labels == -1).sum()

    if n_clusters >= 2:

        mask = labels != -1

        silhouette = silhouette_score(
            Xb_scaled[mask],
            labels[mask]
        )

    else:
        silhouette = np.nan

    dbscan_results.append({
        "eps": eps,
        "clusters": n_clusters,
        "noise": n_noise,
        "noise_pct": n_noise / len(labels) * 100,
        "silhouette": silhouette
    })

dbscan_results = pd.DataFrame(dbscan_results)

print(dbscan_results)

     eps  clusters  noise  noise_pct  silhouette
0 0.5000         3   1601    32.7202      0.2365
1 0.7500         1   1403    28.6736         NaN
2 1.0000         1   1313    26.8343         NaN
3 1.2500         1   1297    26.5073         NaN
4 1.5000         1   1289    26.3438         NaN


In [45]:
# ---------------------------------------------------------
# TASK 8 — ISOLATION FOREST
# ---------------------------------------------------------

from sklearn.ensemble import IsolationForest

contaminations = [0.01, 0.02, 0.05, 0.10]

if_results = []

for contamination in contaminations:

    iso = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=300,
        n_jobs=-1
    )

    labels = iso.fit_predict(Xb_scaled)

    anomaly_score = -iso.score_samples(Xb_scaled)

    n_anomalies = (labels == -1).sum()

    if_results.append({
        "contamination": contamination,
        "anomalies": n_anomalies,
        "anomaly_pct": n_anomalies / len(labels) * 100,
        "score_min": anomaly_score.min(),
        "score_median": np.median(anomaly_score),
        "score_max": anomaly_score.max()
    })

if_results = pd.DataFrame(if_results)

print(if_results)

   contamination  anomalies  anomaly_pct  score_min  score_median  score_max
0         0.0100         49       1.0014     0.3249        0.3487     0.8104
1         0.0200         98       2.0029     0.3249        0.3487     0.8104
2         0.0500        245       5.0072     0.3249        0.3487     0.8104
3         0.1000        490      10.0143     0.3249        0.3487     0.8104


In [46]:
# ---------------------------------------------------------
# TASK 8B — INSPECT ISOLATION FOREST ANOMALIES
# ---------------------------------------------------------

iso = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=300,
    n_jobs=-1
)

if_labels = iso.fit_predict(Xb_scaled)
if_scores = -iso.score_samples(Xb_scaled)

if_analysis = model_df[
    ["event_id",
     "centroid_lat",
     "centroid_lon"] +
    behavior_features
].copy()

if_analysis["anomaly_score"] = if_scores
if_analysis["is_anomaly"] = if_labels == -1

print("Anomalies:", if_analysis["is_anomaly"].sum())

print("\nAnomaly profile:")
print(
    if_analysis[
        if_analysis["is_anomaly"]
    ][behavior_features]
    .describe()
)

print("\nNormal profile:")
print(
    if_analysis[
        ~if_analysis["is_anomaly"]
    ][behavior_features]
    .describe()
)

print("\nTop 20 anomalous events:")
print(
    if_analysis[
        if_analysis["is_anomaly"]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    [
        [
            "event_id",
            "anomaly_score",
            "active_days",
            "max_frp",
            "max_bright_ti4",
            "detection_count",
            "spatial_diameter_km"
        ]
    ]
    .head(20)
)

Anomalies: 245

Anomaly profile:
       mean_frp  max_frp  std_frp  mean_bright_ti4  max_bright_ti4  \
count  245.0000 245.0000 245.0000         245.0000        245.0000   
mean     2.4563   4.2694   1.0692         310.9982        326.1892   
std      2.3708   3.2937   1.3436          11.1014         14.1564   
min      0.6255   1.0900   0.0000         297.2392        300.4300   
25%      1.2811   2.3300   0.4700         304.3000        315.2100   
50%      1.6354   3.3100   0.6705         307.2494        324.3600   
75%      2.4867   4.8200   0.9876         313.7822        334.7700   
max     21.3700  28.6300  10.9963         361.0200        367.0000   

       std_bright_ti4  mean_bright_ti5  max_bright_ti5  std_bright_ti5  \
count        245.0000         245.0000        245.0000        245.0000   
mean           8.9130         285.8009        289.2251          2.7649   
std            6.4031           5.1704          5.9154          2.3158   
min            0.0000         259.6600  

In [47]:
# ---------------------------------------------------------
# TASK 9 — SAVE BEHAVIOUR + ANOMALY OUTPUTS
# ---------------------------------------------------------

# K-Means behavioural cluster
model_df["behavior_cluster"] = behavior_labels

model_df["behavior_label"] = model_df["behavior_cluster"].map({
    0: "Transient",
    1: "Persistent_Elevated"
})

# Isolation Forest
model_df["isolation_score"] = if_scores
model_df["isolation_anomaly"] = if_labels == -1

# Save
model_df.to_csv(
    "viirs_v11_behavior_anomaly_features.csv",
    index=False
)

print("Saved: viirs_v11_behavior_anomaly_features.csv")

print("\nBehavior:")
print(model_df["behavior_label"].value_counts())

print("\nIsolation Forest:")
print(model_df["isolation_anomaly"].value_counts())

Saved: viirs_v11_behavior_anomaly_features.csv

Behavior:
behavior_label
Transient              4500
Persistent_Elevated     393
Name: count, dtype: int64

Isolation Forest:
isolation_anomaly
False    4648
True      245
Name: count, dtype: int64


In [48]:
# ============================================================
# TASK 10 — BEHAVIOR vs COMBINED CONTEXT CLUSTERING
# ============================================================

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 1. Prepare matrices
# ------------------------------------------------------------

X_behavior = model_df[behavior_features].copy()

# Context features = OSM + WorldCover indicators
context_features = [
    c for c in model_df.columns
    if (
        c.startswith("nearest_")
        or "_count_" in c
        or c.startswith("wc_")
    )
    and c != "worldcover_missing"
]

X_context = model_df[context_features].copy()

X_combined = pd.concat(
    [X_behavior, X_context],
    axis=1
)

print("Behavior features :", X_behavior.shape)
print("Context features  :", X_context.shape)
print("Combined features :", X_combined.shape)


# ------------------------------------------------------------
# 2. Remove zero-variance features
# ------------------------------------------------------------

def remove_zero_variance(df):
    variances = df.var()
    keep = variances[variances > 0].index.tolist()
    removed = [c for c in df.columns if c not in keep]
    return df[keep], removed


X_behavior, removed_behavior = remove_zero_variance(X_behavior)
X_context, removed_context = remove_zero_variance(X_context)
X_combined, removed_combined = remove_zero_variance(X_combined)

print("\nZero-variance removed:")
print("Behavior :", len(removed_behavior))
print("Context  :", len(removed_context))
print("Combined :", len(removed_combined))


# ------------------------------------------------------------
# 3. Robust scaling
# ------------------------------------------------------------

behavior_scaled = RobustScaler().fit_transform(X_behavior)
context_scaled = RobustScaler().fit_transform(X_context)
combined_scaled = RobustScaler().fit_transform(X_combined)


# ------------------------------------------------------------
# 4. Compare K-Means
# ------------------------------------------------------------

results = []

for dataset_name, X in [
    ("Behavior", behavior_scaled),
    ("Context", context_scaled),
    ("Combined", combined_scaled)
]:

    for k in range(2, 6):

        km = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=42
        )

        labels = km.fit_predict(X)

        sil = silhouette_score(X, labels)

        results.append({
            "dataset": dataset_name,
            "k": k,
            "inertia": km.inertia_,
            "silhouette": sil
        })

results_df = pd.DataFrame(results)

print("\nK-MEANS COMPARISON")
display(results_df.round(4))

Behavior features : (4893, 19)
Context features  : (4893, 42)
Combined features : (4893, 61)

Zero-variance removed:
Behavior : 0
Context  : 20
Combined : 20

K-MEANS COMPARISON


,dataset,k,inertia,silhouette
0,Behavior,2,20777586.7649,0.8461
1,Behavior,3,14014385.1921,0.8131
2,Behavior,4,11521489.9173,0.8014
3,Behavior,5,9670463.5649,0.8187
4,Context,2,24801.3249,0.4223
5,Context,3,20032.0899,0.3618
6,Context,4,16620.8573,0.2890
7,Context,5,14015.6845,0.2667
8,Combined,2,20809760.0530,0.8427
9,Combined,3,14046410.7348,0.8062


In [49]:
# ============================================================
# TASK 10B — INSPECT BEHAVIOR vs COMBINED CLUSTERS
# ============================================================

# Fit behavior K=2
km_behavior = KMeans(
    n_clusters=2,
    n_init=20,
    random_state=42
)

behavior_labels_2 = km_behavior.fit_predict(behavior_scaled)


# Fit combined K=2
km_combined = KMeans(
    n_clusters=2,
    n_init=20,
    random_state=42
)

combined_labels_2 = km_combined.fit_predict(combined_scaled)


# ------------------------------------------------------------
# Add temporary labels
# ------------------------------------------------------------

comparison_df = model_df.copy()

comparison_df["behavior_cluster_2"] = behavior_labels_2
comparison_df["combined_cluster_2"] = combined_labels_2


# ------------------------------------------------------------
# Compare cluster sizes
# ------------------------------------------------------------

print("BEHAVIOR K=2")
print(
    comparison_df["behavior_cluster_2"]
    .value_counts()
    .sort_index()
)

print("\nCOMBINED K=2")
print(
    comparison_df["combined_cluster_2"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Behavior profiles
# ------------------------------------------------------------

profile_columns = [
    "mean_frp",
    "max_frp",
    "max_bright_ti4",
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km"
]

print("\nBEHAVIOR CLUSTER PROFILES")
display(
    comparison_df
    .groupby("behavior_cluster_2")[profile_columns]
    .mean()
    .round(3)
)


# ------------------------------------------------------------
# Combined cluster behavior profiles
# ------------------------------------------------------------

print("\nCOMBINED CLUSTER BEHAVIOR PROFILES")
display(
    comparison_df
    .groupby("combined_cluster_2")[profile_columns]
    .mean()
    .round(3)
)


# ------------------------------------------------------------
# WorldCover composition
# ------------------------------------------------------------

wc_columns = [
    "wc_tree_cover",
    "wc_shrubland",
    "wc_grassland",
    "wc_cropland",
    "wc_built_up",
    "wc_bare_sparse",
    "wc_permanent_water"
]

print("\nCOMBINED CLUSTER WORLDCOVER PROFILES")
display(
    comparison_df
    .groupby("combined_cluster_2")[wc_columns]
    .mean()
    .round(3)
)


# ------------------------------------------------------------
# OSM context
# ------------------------------------------------------------

osm_count_columns = [
    c for c in context_features
    if "_count_" in c
]

print("\nCOMBINED CLUSTER OSM PROFILES")
display(
    comparison_df
    .groupby("combined_cluster_2")[osm_count_columns]
    .mean()
    .round(4)
)

BEHAVIOR K=2
behavior_cluster_2
0    4500
1     393
Name: count, dtype: int64

COMBINED K=2
combined_cluster_2
0    4501
1     392
Name: count, dtype: int64

BEHAVIOR CLUSTER PROFILES


,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km
behavior_cluster_2,,,,,,,,
0,1.3100,1.3700,305.4790,1.1790,1.2590,1.3330,1.2020,0.0520
1,1.5770,2.7810,318.0150,8.1270,10.2160,19.3610,15.9540,0.6600



COMBINED CLUSTER BEHAVIOR PROFILES


,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km
combined_cluster_2,,,,,,,,
0,1.3110,1.3700,305.4780,1.1790,1.2590,1.3340,1.2020,0.0520
1,1.5760,2.7820,318.0580,8.1450,10.2400,19.4010,15.9900,0.6600



COMBINED CLUSTER WORLDCOVER PROFILES


,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
combined_cluster_2,,,,,,,
0,0.3030,0.0320,0.1900,0.1870,0.1150,0.0840,0.0100
1,0.1400,0.0130,0.0940,0.0610,0.2930,0.3420,0.0310



COMBINED CLUSTER OSM PROFILES


,industrial_zone_count_375m,industrial_zone_count_1km,industrial_zone_count_3km,factory_count_375m,factory_count_1km,factory_count_3km,mine_count_375m,mine_count_1km,mine_count_3km,brick_count_375m,brick_count_1km,brick_count_3km,works_count_375m,works_count_1km,works_count_3km,depot_count_375m,depot_count_1km,depot_count_3km,power_count_375m,power_count_1km,power_count_3km,other_industry_count_375m,other_industry_count_1km,other_industry_count_3km
combined_cluster_2,,,,,,,,,,,,,,,,,,,,,,,,
0,0.0000,0.0000,0.0013,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0011,0.0038,0.0338,0.0000,0.0000,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0009
1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0026,0.0102,0.0842,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [50]:
# ============================================================
# TASK 11 — PREPARE VALIDATION EVALUATION
# ============================================================

import os
import pandas as pd
import numpy as np

VALIDATION_FILE = "viirs_validation_candidates_v11.csv"

print("Looking for validation file...")

if os.path.exists(VALIDATION_FILE):
    validation = pd.read_csv(VALIDATION_FILE)

    print("Validation file found.")
    print("Shape:", validation.shape)
    print("\nColumns:")
    print(validation.columns.tolist())

    print("\nClass labels:")
    if "actual_class" in validation.columns:
        print(validation["actual_class"].value_counts(dropna=False))
else:
    print("Validation file NOT found.")
    print("Expected:", VALIDATION_FILE)

Looking for validation file...
Validation file NOT found.
Expected: viirs_validation_candidates_v11.csv


In [51]:
# ============================================================
# TASK 11A — VALIDATION SAMPLE READINESS CHECK
# ============================================================

VALIDATION_SAMPLE_SIZE = 60

print("Total events available:", len(model_df))

# ------------------------------------------------------------
# Check whether an existing validation sample is already
# available in the current dataframe / notebook state
# ------------------------------------------------------------

candidate_ids = None

for name in [
    "validation_candidates",
    "validation_sample",
    "validation_df"
]:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame) and "event_id" in obj.columns:
            candidate_ids = set(obj["event_id"].dropna().astype(int))
            print(f"\nFound existing object: {name}")
            print("Rows:", len(obj))
            print("Unique event IDs:", len(candidate_ids))
            break

if candidate_ids is None:
    print("\nNo validation sample dataframe currently loaded.")

# ------------------------------------------------------------
# Check event dataset uniqueness / integrity
# ------------------------------------------------------------

print("\nEVENT DATASET INTEGRITY")
print("Unique event IDs:", model_df["event_id"].nunique())
print("Duplicate event IDs:", model_df["event_id"].duplicated().sum())

# ------------------------------------------------------------
# Show useful candidate groups for validation
# ------------------------------------------------------------

print("\nPERSISTENT EVENTS")
persistent = model_df[model_df["active_days"] >= 15]

print("Count:", len(persistent))
display(
    persistent[
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "active_days",
            "duration_days",
            "detection_count",
            "spatial_diameter_km",
            "max_frp",
            "max_bright_ti4"
        ]
    ]
    .sort_values("active_days", ascending=False)
    .head(15)
)

print("\nSPATIALLY EXPANDED EVENTS (>375 m)")
expanded = model_df[model_df["spatial_diameter_km"] > 0.375]

print("Count:", len(expanded))
display(
    expanded[
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "active_days",
            "duration_days",
            "detection_count",
            "spatial_diameter_km",
            "max_frp",
            "max_bright_ti4"
        ]
    ]
    .sort_values("spatial_diameter_km", ascending=False)
    .head(15)
)

print("\nSHORT + HIGH FRP EVENTS")
short_high = model_df[
    (model_df["active_days"] <= 3) &
    (model_df["max_frp"] >= 2)
]

print("Count:", len(short_high))
display(
    short_high[
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "active_days",
            "duration_days",
            "detection_count",
            "spatial_diameter_km",
            "max_frp",
            "max_bright_ti4"
        ]
    ]
    .sort_values("max_frp", ascending=False)
    .head(15)
)

Total events available: 4893

No validation sample dataframe currently loaded.

EVENT DATASET INTEGRITY
Unique event IDs: 4893
Duplicate event IDs: 0

PERSISTENT EVENTS
Count: 73


,event_id,centroid_lat,centroid_lon,active_days,duration_days,detection_count,spatial_diameter_km,max_frp,max_bright_ti4
48,48,21.1057,72.6407,31,31,205,2.6092,15.6900,354.2000
40,40,21.4870,81.7687,29,31,46,0.6968,2.6700,318.0600
43,43,18.6850,73.0361,28,31,94,1.8755,4.8200,340.4000
2,2,22.0539,88.1241,27,31,85,1.0219,3.9000,330.2000
0,0,23.1695,82.3411,27,31,85,1.5451,4.9700,337.4800
36,36,15.1715,76.3785,27,31,56,1.1537,3.1500,317.5700
18,18,22.0420,83.7352,27,31,113,1.4593,6.3400,356.5200
17,17,23.7697,86.3942,27,31,694,3.7180,5.1300,343.5600
127,127,15.1759,76.6577,26,31,86,2.5376,5.0800,344.2900
180,180,23.7809,86.3455,26,31,65,1.2844,4.5100,331.9100



SPATIALLY EXPANDED EVENTS (>375 m)
Count: 402


,event_id,centroid_lat,centroid_lon,active_days,duration_days,detection_count,spatial_diameter_km,max_frp,max_bright_ti4
17,17,23.7697,86.3942,27,31,694,3.7180,5.1300,343.5600
48,48,21.1057,72.6407,31,31,205,2.6092,15.6900,354.2000
127,127,15.1759,76.6577,26,31,86,2.5376,5.0800,344.2900
80,80,21.7628,84.0212,19,21,125,2.4513,5.0100,331.2400
135,135,20.9664,85.1727,16,17,177,2.4327,4.6700,342.1200
19,19,20.7907,85.2585,17,18,97,2.2939,8.9800,346.7600
25,25,23.5561,87.2407,14,14,63,2.2673,4.0400,335.2200
91,91,23.6916,87.1168,15,15,99,2.2545,3.3000,317.0300
2899,2899,23.6919,87.1179,9,13,49,2.2334,4.2100,313.1100
193,193,23.7776,86.2056,26,31,159,2.1293,7.3400,348.6000



SHORT + HIGH FRP EVENTS
Count: 794


,event_id,centroid_lat,centroid_lon,active_days,duration_days,detection_count,spatial_diameter_km,max_frp,max_bright_ti4
4016,4016,30.9375,75.6004,2,2,6,0.7260,28.6300,367.0000
867,867,22.0679,88.1212,1,1,1,0.0000,21.3700,298.1000
893,893,22.0644,88.1207,1,1,1,0.0000,21.3700,344.2300
3038,3038,33.0399,74.9502,1,1,1,0.0000,17.5400,342.7500
3039,3039,33.0406,74.9450,1,1,1,0.0000,17.5400,327.7400
416,416,22.5106,88.3325,1,1,2,0.1921,15.0100,367.0000
4039,4039,30.9434,75.6022,2,2,3,0.2407,14.8400,367.0000
3192,3192,24.0692,69.5759,1,1,1,0.0000,13.7700,304.7400
3069,3069,24.0698,69.5833,1,1,4,0.4891,13.7700,367.0000
2605,2605,27.3422,92.5184,1,1,1,0.0000,12.7700,338.9900


In [52]:
# ============================================================
# TASK 11B — K-MEANS vs ISOLATION FOREST OVERLAP
# ============================================================

# ------------------------------------------------------------
# 1. Ensure required labels exist
# ------------------------------------------------------------

if "behavior_cluster_2" not in model_df.columns:
    model_df["behavior_cluster_2"] = behavior_labels_2

if "isolation_anomaly" not in model_df.columns:
    model_df["isolation_anomaly"] = if_labels == -1

# Persistent/elevated cluster = cluster with higher median
cluster_medians = (
    model_df
    .groupby("behavior_cluster_2")["active_days"]
    .median()
)

persistent_cluster = cluster_medians.idxmax()

model_df["behavior_label_2"] = np.where(
    model_df["behavior_cluster_2"] == persistent_cluster,
    "Persistent_Elevated",
    "Transient"
)

# ------------------------------------------------------------
# 2. Basic overlap
# ------------------------------------------------------------

print("BEHAVIOR CLUSTERS")
print(model_df["behavior_label_2"].value_counts())

print("\nISOLATION FOREST")
print(model_df["isolation_anomaly"].value_counts())

print("\nCROSS-TABULATION")
overlap = pd.crosstab(
    model_df["behavior_label_2"],
    model_df["isolation_anomaly"],
    margins=True
)

display(overlap)


# ------------------------------------------------------------
# 3. Isolation Forest anomalies by behavior
# ------------------------------------------------------------

print("\nISOLATION FOREST ANOMALIES BY BEHAVIOR")

anomaly_by_behavior = (
    model_df[model_df["isolation_anomaly"]]
    .groupby("behavior_label_2")
    .size()
    .reset_index(name="anomaly_count")
)

display(anomaly_by_behavior)


# ------------------------------------------------------------
# 4. Isolation Forest anomaly rate within each behavior
# ------------------------------------------------------------

rate_table = (
    model_df
    .groupby("behavior_label_2")["isolation_anomaly"]
    .agg(
        total_events="count",
        anomalies="sum",
        anomaly_rate="mean"
    )
    .reset_index()
)

rate_table["anomaly_rate"] *= 100

print("\nANOMALY RATE BY BEHAVIOR")
display(rate_table.round(2))


# ------------------------------------------------------------
# 5. Profile the four important groups
# ------------------------------------------------------------

model_df["behavior_if_group"] = np.select(
    [
        (model_df["behavior_label_2"] == "Persistent_Elevated") &
        (model_df["isolation_anomaly"]),

        (model_df["behavior_label_2"] == "Persistent_Elevated") &
        (~model_df["isolation_anomaly"]),

        (model_df["behavior_label_2"] == "Transient") &
        (model_df["isolation_anomaly"]),

        (model_df["behavior_label_2"] == "Transient") &
        (~model_df["isolation_anomaly"])
    ],
    [
        "Persistent + Anomalous",
        "Persistent + Normal",
        "Transient + Anomalous",
        "Transient + Normal"
    ],
    default="Other"
)

profile_cols = [
    "mean_frp",
    "max_frp",
    "max_bright_ti4",
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km"
]

print("\nBEHAVIOR × ISOLATION FOREST PROFILES")

display(
    model_df
    .groupby("behavior_if_group")[profile_cols]
    .agg(["count", "mean", "median"])
    .round(3)
)

BEHAVIOR CLUSTERS
behavior_label_2
Transient              4500
Persistent_Elevated     393
Name: count, dtype: int64

ISOLATION FOREST
isolation_anomaly
False    4648
True      245
Name: count, dtype: int64

CROSS-TABULATION


isolation_anomaly,False,True,All
behavior_label_2,,,
Persistent_Elevated,181,212,393
Transient,4467,33,4500
All,4648,245,4893



ISOLATION FOREST ANOMALIES BY BEHAVIOR


,behavior_label_2,anomaly_count
0,Persistent_Elevated,212
1,Transient,33



ANOMALY RATE BY BEHAVIOR


,behavior_label_2,total_events,anomalies,anomaly_rate
0,Persistent_Elevated,393,212,53.9400
1,Transient,4500,33,0.7300



BEHAVIOR × ISOLATION FOREST PROFILES


mean_frp               max_frp                \
                          count   mean median   count   mean median   
behavior_if_group                                                     
Persistent + Anomalous      212 1.9790 1.6060     212 3.8620 3.1400   
Persistent + Normal         181 1.1060 1.0100     181 1.5160 1.4400   
Transient + Anomalous        33 5.5220 4.5200      33 6.8880 6.2600   
Transient + Normal         4467 1.2790 1.0100    4467 1.3290 1.0500   

                       max_bright_ti4                   active_days          \
                                count     mean   median       count    mean   
behavior_if_group                                                             
Persistent + Anomalous            212 325.2990 323.1750         212 11.6510   
Persistent + Normal               181 309.4840 307.9300         181  4.0000   
Transient + Anomalous              33 331.9100 331.1700          33  2.1820   
Transient + Normal               4467 305.2840 302.7500        4467  1.1710   

                               duration_days                 detection_count  \
                        median         count    mean  median           count   
behavior_if_group                                                              
Persistent + Anomalous 10.0000           212 14.3210 14.0000             212   
Persistent + Normal     3.0000           181  5.4090  5.0000             181   
Transient + Anomalous   1.0000            33  2.9090  1.0000              33   
Transient + Normal      1.0000          4467  1.2470  1.0000            4467   

                                       daily_object_count                  \
                          mean  median              count    mean  median   
behavior_if_group                                                           
Persistent + Anomalous 31.6700 18.5000                212 25.8350 16.0000   
Persistent + Normal     4.9450  4.0000                181  4.3810  4.0000   
Transient + Anomalous   4.0610  2.0000                 33  3.0300  1.0000   
Transient + Normal      1.3130  1.0000               4467  1.1880  1.0000   

                       spatial_diameter_km                
                                     count   mean median  
behavior_if_group                                         
Persistent + Anomalous                 212 0.8880 0.7150  
Persistent + Normal                    181 0.3920 0.3740  
Transient + Anomalous                   33 0.3240 0.2150  
Transient + Normal                    4467 0.0500 0.0000

In [53]:
# ============================================================
# TASK 11C — BEHAVIORAL GROUP × GEOGRAPHIC CONTEXT
# ============================================================

# ------------------------------------------------------------
# Context columns
# ------------------------------------------------------------

context_summary_cols = [
    "wc_built_up",
    "wc_cropland",
    "wc_tree_cover",
    "wc_grassland",
    "wc_bare_sparse",

    "entity_works_count_375m",
    "entity_works_count_1km",
    "entity_works_count_3km",

    "entity_industrial_zone_count_3km",
    "entity_factory_count_3km",
    "entity_mine_count_3km",
    "entity_brick_count_3km",
    "entity_depot_count_3km",
    "entity_power_count_3km",
    "entity_other_industry_count_3km"
]

# Keep only columns that actually exist
context_summary_cols = [
    c for c in context_summary_cols
    if c in model_df.columns
]

print("Context columns used:")
print(context_summary_cols)


# ------------------------------------------------------------
# Group profiles
# ------------------------------------------------------------

print("\nBEHAVIORAL GROUP × GEOGRAPHIC CONTEXT")

display(
    model_df
    .groupby("behavior_if_group")[context_summary_cols]
    .mean()
    .round(4)
)


# ------------------------------------------------------------
# Number of events with OSM evidence
# ------------------------------------------------------------

osm_presence_cols = [
    c for c in context_summary_cols
    if "_count_" in c
]

osm_presence = (
    model_df[osm_presence_cols]
    .gt(0)
    .any(axis=1)
)

model_df["has_osm_context"] = osm_presence


osm_by_group = (
    model_df
    .groupby("behavior_if_group")["has_osm_context"]
    .agg(
        total_events="count",
        events_with_osm="sum",
        proportion_with_osm="mean"
    )
    .reset_index()
)

osm_by_group["proportion_with_osm"] *= 100

print("\nOSM COVERAGE BY BEHAVIORAL GROUP")

display(osm_by_group.round(2))


# ------------------------------------------------------------
# WorldCover distribution by group
# ------------------------------------------------------------

wc_cols = [
    "wc_tree_cover",
    "wc_shrubland",
    "wc_grassland",
    "wc_cropland",
    "wc_built_up",
    "wc_bare_sparse",
    "wc_permanent_water"
]

wc_cols = [c for c in wc_cols if c in model_df.columns]

print("\nWORLD COVER BY BEHAVIORAL GROUP")

display(
    model_df
    .groupby("behavior_if_group")[wc_cols]
    .mean()
    .round(4)
)

Context columns used:
['wc_built_up', 'wc_cropland', 'wc_tree_cover', 'wc_grassland', 'wc_bare_sparse']

BEHAVIORAL GROUP × GEOGRAPHIC CONTEXT


,wc_built_up,wc_cropland,wc_tree_cover,wc_grassland,wc_bare_sparse
behavior_if_group,,,,,
Persistent + Anomalous,0.2500,0.0613,0.1321,0.0943,0.4009
Persistent + Normal,0.3425,0.0608,0.1492,0.0994,0.2707
Transient + Anomalous,0.1818,0.1818,0.3030,0.1212,0.0303
Transient + Normal,0.1144,0.1874,0.3029,0.1898,0.0842



OSM COVERAGE BY BEHAVIORAL GROUP


,behavior_if_group,total_events,events_with_osm,proportion_with_osm
0,Persistent + Anomalous,212,0,0.0000
1,Persistent + Normal,181,0,0.0000
2,Transient + Anomalous,33,0,0.0000
3,Transient + Normal,4467,0,0.0000



WORLD COVER BY BEHAVIORAL GROUP


,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
behavior_if_group,,,,,,,
Persistent + Anomalous,0.1321,0.0142,0.0943,0.0613,0.2500,0.4009,0.0330
Persistent + Normal,0.1492,0.0110,0.0994,0.0608,0.3425,0.2707,0.0276
Transient + Anomalous,0.3030,0.0000,0.1212,0.1818,0.1818,0.0303,0.0000
Transient + Normal,0.3029,0.0322,0.1898,0.1874,0.1144,0.0842,0.0099


In [54]:
# ============================================================
# TASK 11C — OSM COLUMN / COVERAGE CHECK
# ============================================================

osm_columns = [
    c for c in model_df.columns
    if "entity_" in c and "_count_" in c
]

print("OSM COUNT COLUMNS:")
for c in osm_columns:
    print(c)

print("\nNON-ZERO EVENT COUNTS:")
for c in osm_columns:
    count = (model_df[c] > 0).sum()
    if count > 0:
        print(f"{c:45s} {count}")

print("\nTOTAL EVENTS WITH ANY OSM COUNT:")

if osm_columns:
    any_osm = model_df[osm_columns].gt(0).any(axis=1)
    print(any_osm.sum(), "/", len(model_df))
    print("Percentage:", round(any_osm.mean() * 100, 2), "%")
else:
    print("No OSM count columns found.")

OSM COUNT COLUMNS:

NON-ZERO EVENT COUNTS:

TOTAL EVENTS WITH ANY OSM COUNT:
No OSM count columns found.


In [55]:
# ============================================================
# TASK 11D — INSPECT CURRENT GEOGRAPHIC FEATURES
# ============================================================

geo_cols = [
    c for c in model_df.columns
    if (
        c.startswith("entity_")
        or c.startswith("nearest_")
        or c.startswith("wc_")
    )
]

print("Geographic/context columns currently in model_df:")
print("Count:", len(geo_cols))

for c in geo_cols:
    print(c)

Geographic/context columns currently in model_df:
Count: 18
nearest_industrial_zone_km
nearest_factory_km
nearest_mine_km
nearest_brick_km
nearest_works_km
nearest_depot_km
nearest_other_industry_km
wc_tree_cover
wc_shrubland
wc_grassland
wc_cropland
wc_built_up
wc_bare_sparse
wc_snow_ice
wc_permanent_water
wc_wetland
wc_mangroves
wc_moss_lichen


In [56]:
# ============================================================
# TASK 11D — BEHAVIORAL GROUP × OSM DISTANCE
# ============================================================

nearest_cols = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_works_km",
    "nearest_depot_km",
    "nearest_other_industry_km"
]

nearest_cols = [
    c for c in nearest_cols
    if c in model_df.columns
]

print("OSM distance features:")
print(nearest_cols)

print("\nOSM DISTANCE BY BEHAVIORAL GROUP")

display(
    model_df
    .groupby("behavior_if_group")[nearest_cols]
    .agg(["count", "median", "mean", "min"])
    .round(3)
)

OSM distance features:
['nearest_industrial_zone_km', 'nearest_factory_km', 'nearest_mine_km', 'nearest_brick_km', 'nearest_works_km', 'nearest_depot_km', 'nearest_other_industry_km']

OSM DISTANCE BY BEHAVIORAL GROUP


nearest_industrial_zone_km                            \
                                            count   median     mean     min   
behavior_if_group                                                             
Persistent + Anomalous                        212 123.3810 122.7730  3.2230   
Persistent + Normal                           181 128.0400 142.2030  3.8120   
Transient + Anomalous                          33 211.6200 250.2380 13.5910   
Transient + Normal                           4467 158.0530 175.0250  2.0770   

                       nearest_factory_km                             \
                                    count   median     mean      min   
behavior_if_group                                                      
Persistent + Anomalous                212 279.7870 344.8980  15.8420   
Persistent + Normal                   181 299.2020 371.5470  15.2090   
Transient + Anomalous                  33 413.9930 513.0870 230.1050   
Transient + Normal                   4467 303.8770 370.6690   6.7400   

                       nearest_mine_km                               \
                                 count    median      mean      min   
behavior_if_group                                                     
Persistent + Anomalous             212  741.8900  917.3600 161.6070   
Persistent + Normal                181  949.4100 1121.7470 351.9610   
Transient + Anomalous               33 1406.8660 1285.5080 322.6700   
Transient + Normal                4467 1385.4170 1350.6170 115.4640   

                       nearest_brick_km                             \
                                  count   median     mean      min   
behavior_if_group                                                    
Persistent + Anomalous              212 591.3530 684.7830 104.7020   
Persistent + Normal                 181 655.1950 656.5170  38.0800   
Transient + Anomalous                33 448.8570 594.1350 122.3160   
Transient + Normal                 4467 402.6970 478.0790   5.5980   

                       nearest_works_km                         \
                                  count  median    mean    min   
behavior_if_group                                                
Persistent + Anomalous              212 42.1910 49.3750 0.2970   
Persistent + Normal                 181 44.1310 50.6750 0.9780   
Transient + Anomalous                33 52.5460 52.6580 2.6330   
Transient + Normal                 4467 49.8910 56.8230 0.1300   

                       nearest_depot_km                            \
                                  count   median     mean     min   
behavior_if_group                                                   
Persistent + Anomalous              212 422.7060 407.8990  3.2230   
Persistent + Normal                 181 475.6790 438.0660  3.8120   
Transient + Anomalous                33 334.7660 370.6660 96.0940   
Transient + Normal                 4467 291.6310 330.8170  2.2160   

                       nearest_other_industry_km                            
                                           count   median     mean     min  
behavior_if_group                                                           
Persistent + Anomalous                       212 290.9830 299.6730  9.6840  
Persistent + Normal                          181 200.3770 240.0160 16.0150  
Transient + Anomalous                         33 386.9870 443.6330 13.5910  
Transient + Normal                          4467 263.2640 283.4760  2.0770

In [57]:
# ============================================================
# TASK 12 — VALIDATION MERGE READINESS
# ============================================================

# ------------------------------------------------------------
# Verify the frozen event dataset
# ------------------------------------------------------------

print("Frozen V11 events:", len(model_df))
print("Unique event IDs:", model_df["event_id"].nunique())

# ------------------------------------------------------------
# Check the model outputs we will evaluate
# ------------------------------------------------------------

required_model_columns = [
    "event_id",
    "behavior_label_2",
    "isolation_score",
    "isolation_anomaly"
]

print("\nRequired model outputs:")

for col in required_model_columns:
    print(
        f"{col:25s}",
        "OK" if col in model_df.columns else "MISSING"
    )

# ------------------------------------------------------------
# Check current classification if available
# ------------------------------------------------------------

classification_cols = [
    "final_class",
    "classification",
    "predicted_class",
    "domain_class"
]

print("\nPossible classification columns:")

for col in classification_cols:
    if col in model_df.columns:
        print(col, "FOUND")

# ------------------------------------------------------------
# Save a clean validation lookup table
# ------------------------------------------------------------

validation_lookup_cols = [
    "event_id",
    "centroid_lat",
    "centroid_lon",
    "start_date",
    "end_date",
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "spatial_diameter_km",
    "mean_frp",
    "max_frp",
    "max_bright_ti4",
    "behavior_label_2",
    "isolation_score",
    "isolation_anomaly"
]

validation_lookup_cols = [
    c for c in validation_lookup_cols
    if c in model_df.columns
]

validation_lookup = model_df[validation_lookup_cols].copy()

validation_lookup.to_csv(
    "viirs_v11_validation_model_lookup.csv",
    index=False
)

print("\nSaved:")
print("viirs_v11_validation_model_lookup.csv")

print("\nShape:", validation_lookup.shape)
display(validation_lookup.head())

Frozen V11 events: 4893
Unique event IDs: 4893

Required model outputs:
event_id                  OK
behavior_label_2          OK
isolation_score           OK
isolation_anomaly         OK

Possible classification columns:

Saved:
viirs_v11_validation_model_lookup.csv

Shape: (4893, 14)


,event_id,centroid_lat,centroid_lon,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,max_bright_ti4,behavior_label_2,isolation_score,isolation_anomaly
0,0,23.1695,82.3411,27,31,85,75,1.5451,1.6354,4.9700,337.4800,Persistent_Elevated,0.7486,True
1,1,24.2093,82.7125,2,2,3,2,0.3679,1.4333,2.3200,306.6300,Transient,0.4689,False
2,2,22.0539,88.1241,27,31,85,64,1.0219,1.7355,3.9000,330.2000,Persistent_Elevated,0.7287,True
3,3,24.2042,82.7115,2,2,3,2,0.3496,1.4367,2.0700,307.3100,Transient,0.4519,False
4,4,22.3196,82.5665,1,1,2,1,0.3579,0.8000,0.8000,302.9700,Transient,0.4103,False
